# Assignment 3: Foundation Model-Based One-Shot Image Classification

**Objective:** Evaluate foundation models for 16-class image classification using exactly one labeled training image per class.

**Dataset:** 2,400 training images and 400 test images.

**Evaluation:** Validation accuracy for experiment development and test accuracy for final performance reporting.

## 1. GPU Verification and Library Imports

In [19]:
import os
import gc
import json
import time
import random
import hashlib
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from google.colab import files
from IPython.display import display

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Enable T4 GPU under Runtime > Change runtime type."
)

DEVICE = torch.device("cuda")

print("PyTorch Version:", torch.__version__)
print("Torchvision Version:", torchvision.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA Version:", torch.version.cuda)
print(
    "GPU Memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

PyTorch Version: 2.14.1+cu130
Torchvision Version: 0.29.1+cu130
CUDA Available: True
GPU: Tesla T4
CUDA Version: 13.0
GPU Memory: 14.56 GB


## 2. Verifying Uploaded Train and Test Datasets

In [2]:
required_files = ["train.zip", "test.zip"]

for filename in required_files:
    filepath = f"/content/{filename}"

    assert os.path.exists(filepath), f"Missing file: {filename}"

    size_mb = os.path.getsize(filepath) / (1024 ** 2)
    print(f"{filename}: {size_mb:.2f} MB")

print("\nBoth dataset ZIP files are available.")
print("Ready for dataset extraction.")

train.zip: 48.08 MB
test.zip: 7.80 MB

Both dataset ZIP files are available.
Ready for dataset extraction.


## 3. Dataset Extraction and Class Verification

In [3]:
ROOT = Path("/content/assignment3")
TRAIN_DIR = ROOT / "train"
TEST_DIR = ROOT / "test"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def extract_dataset(zip_file, destination):
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_file, "r") as archive:
        assert archive.testzip() is None, f"Corrupted ZIP: {zip_file}"

        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target.is_relative_to(destination.resolve()), (
                f"Unsafe ZIP path: {member.filename}"
            )

        archive.extractall(destination)

    print(f"Extracted {zip_file}")


extract_dataset("/content/train.zip", TRAIN_DIR)
extract_dataset("/content/test.zip", TEST_DIR)


def find_class_directory(root):
    """
    Locate a directory containing the 16 class folders.
    Supports ZIPs with an additional enclosing folder.
    """
    candidates = [root] + sorted(
        p for p in root.rglob("*") if p.is_dir()
    )

    for candidate in candidates:
        class_folders = [
            p for p in candidate.iterdir()
            if p.is_dir()
            and any(
                f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
                for f in p.iterdir()
            )
        ]

        if len(class_folders) == 16:
            return candidate

    raise RuntimeError(f"Could not locate 16 class folders in {root}")


TRAIN_CLASSES_DIR = find_class_directory(TRAIN_DIR)
TEST_CLASSES_DIR = find_class_directory(TEST_DIR)


def collect_images(directory):
    data = {}

    for class_folder in sorted(directory.iterdir()):
        if not class_folder.is_dir():
            continue

        images = sorted(
            p for p in class_folder.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )

        if images:
            data[class_folder.name] = images

    return data


train_by_class = collect_images(TRAIN_CLASSES_DIR)
test_by_class = collect_images(TEST_CLASSES_DIR)

CLASS_NAMES = sorted(train_by_class)
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}

assert len(CLASS_NAMES) == 16, "Expected 16 classes."
assert set(train_by_class) == set(test_by_class), "Class names differ."
assert all(len(v) == 150 for v in train_by_class.values())
assert all(len(v) == 25 for v in test_by_class.values())

class_summary = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Training Images": [len(train_by_class[c]) for c in CLASS_NAMES],
    "Test Images": [len(test_by_class[c]) for c in CLASS_NAMES]
})

display(class_summary)

print("Total Training Images:", sum(map(len, train_by_class.values())))
print("Total Test Images:", sum(map(len, test_by_class.values())))
print("Dataset verification successful.")

Extracted /content/train.zip
Extracted /content/test.zip


,Class,Training Images,Test Images
0,Bedroom,150,25
1,Coast,150,25
2,Flower,150,25
3,Forest,150,25
4,Highway,150,25
5,Industrial,150,25
6,InsideCity,150,25
7,Kitchen,150,25
8,LivingRoom,150,25
9,Mountain,150,25


Total Training Images: 2400
Total Test Images: 400
Dataset verification successful.


## 4. One-Shot Training and Validation Split

In [4]:
rng = random.Random(SEED)

# Selecting exactly one image per class from the original training set.
ONE_SHOT_BY_CLASS = {
    class_name: rng.choice(train_by_class[class_name])
    for class_name in CLASS_NAMES
}

ONE_SHOT_SAMPLES = [
    (ONE_SHOT_BY_CLASS[c], CLASS_TO_IDX[c])
    for c in CLASS_NAMES
]

# Excluding the selected training image from validation.
VALIDATION_SAMPLES = [
    (path, CLASS_TO_IDX[c])
    for c in CLASS_NAMES
    for path in train_by_class[c]
    if path != ONE_SHOT_BY_CLASS[c]
]

# Test images come only from the original test ZIP.
TEST_SAMPLES = [
    (path, CLASS_TO_IDX[c])
    for c in CLASS_NAMES
    for path in test_by_class[c]
]

split_summary = pd.DataFrame({
    "Dataset": ["One-Shot Training", "Validation", "Testing"],
    "Images Per Class": [1, 149, 25],
    "Total Images": [
        len(ONE_SHOT_SAMPLES),
        len(VALIDATION_SAMPLES),
        len(TEST_SAMPLES)
    ]
})

display(split_summary)

print("\nSelected One-Shot Training Images:")

for class_name, image_path in ONE_SHOT_BY_CLASS.items():
    print(f"{class_name}: {image_path.name}")

,Dataset,Images Per Class,Total Images
0,One-Shot Training,1,16
1,Validation,149,2384
2,Testing,25,400



Selected One-Shot Training Images:
Bedroom: image_0029.jpg
Coast: image_0007.jpg
Flower: image_0121.jpg
Forest: image_0063.jpg
Highway: image_0058.jpg
Industrial: image_0036.jpg
InsideCity: image_0027.jpg
Kitchen: image_0188.jpg
LivingRoom: image_0023.jpg
Mountain: image_0238.jpg
Office: image_0009.jpg
OpenCountry: image_0008.jpg
Store: image_0024.jpg
Street: image_0056.jpg
Suburb: image_0060.jpg
TallBuilding: image_0294.jpg


## 5. Dataset Integrity and Overlap Verification

In [5]:
train_paths = {p.resolve() for p, _ in ONE_SHOT_SAMPLES}
val_paths = {p.resolve() for p, _ in VALIDATION_SAMPLES}
test_paths = {p.resolve() for p, _ in TEST_SAMPLES}

assert len(ONE_SHOT_SAMPLES) == 16
assert len(VALIDATION_SAMPLES) == 2384
assert len(TEST_SAMPLES) == 400

assert train_paths.isdisjoint(val_paths), "TRAIN/VALIDATION OVERLAP!"
assert train_paths.isdisjoint(test_paths), "TRAIN/TEST OVERLAP!"
assert val_paths.isdisjoint(test_paths), "VALIDATION/TEST OVERLAP!"

assert Counter(y for _, y in ONE_SHOT_SAMPLES) == {
    i: 1 for i in range(16)
}

assert Counter(y for _, y in VALIDATION_SAMPLES) == {
    i: 149 for i in range(16)
}

assert Counter(y for _, y in TEST_SAMPLES) == {
    i: 25 for i in range(16)
}


def calculate_hash(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


split_hashes = {}

for split_name, samples in {
    "Training": ONE_SHOT_SAMPLES,
    "Validation": VALIDATION_SAMPLES,
    "Testing": TEST_SAMPLES
}.items():
    split_hashes[split_name] = {
        calculate_hash(path) for path, _ in samples
    }

assert split_hashes["Training"].isdisjoint(split_hashes["Validation"])
assert split_hashes["Training"].isdisjoint(split_hashes["Testing"])
assert split_hashes["Validation"].isdisjoint(split_hashes["Testing"])

print("PASS: Exactly one training image per class.")
print("PASS: Exactly 149 validation images per class.")
print("PASS: Exactly 25 test images per class.")
print("PASS: No overlapping image paths.")
print("PASS: No exact duplicate image contents across splits.")
print("\nALL DATASET INTEGRITY CHECKS PASSED.")

PASS: Exactly one training image per class.
PASS: Exactly 149 validation images per class.
PASS: Exactly 25 test images per class.
PASS: No overlapping image paths.
PASS: No exact duplicate image contents across splits.

ALL DATASET INTEGRITY CHECKS PASSED.


## 6. Reusable Dataset and DataLoader Functions

In [6]:
class SceneDataset(Dataset):

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_path, label = self.samples[index]

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return image, label


def create_loader(samples, transform, batch_size=32):
    dataset = SceneDataset(samples, transform)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )


def calculate_accuracy(predictions, targets):
    predictions = np.asarray(predictions)
    targets = np.asarray(targets)

    assert predictions.shape == targets.shape

    return 100 * np.mean(predictions == targets)


def cleanup_gpu():
    gc.collect()
    torch.cuda.empty_cache()


print("Dataset and DataLoader functions initialized.")
print("Number of Classes:", len(CLASS_NAMES))
print("GPU Device:", DEVICE)

Dataset and DataLoader functions initialized.
Number of Classes: 16
GPU Device: cuda


## 7. Saving Dataset Split Information

In [7]:
split_records = []

for split_name, samples in [
    ("Training", ONE_SHOT_SAMPLES),
    ("Validation", VALIDATION_SAMPLES),
    ("Testing", TEST_SAMPLES)
]:
    for image_path, label in samples:
        split_records.append({
            "Split": split_name,
            "Class": CLASS_NAMES[label],
            "Image": str(image_path.relative_to(ROOT))
        })

split_df = pd.DataFrame(split_records)

split_df.to_csv(ROOT / "dataset_split.csv", index=False)

split_manifest = {
    "random_seed": SEED,
    "class_names": CLASS_NAMES,
    "one_shot_training_images": {
        c: str(p.relative_to(ROOT))
        for c, p in ONE_SHOT_BY_CLASS.items()
    },
    "training_count": len(ONE_SHOT_SAMPLES),
    "validation_count": len(VALIDATION_SAMPLES),
    "test_count": len(TEST_SAMPLES)
}

with open(ROOT / "dataset_split_manifest.json", "w") as f:
    json.dump(split_manifest, f, indent=2)

print("Dataset split information saved.")
print("Random Seed:", SEED)
print("Training:", len(ONE_SHOT_SAMPLES))
print("Validation:", len(VALIDATION_SAMPLES))
print("Testing:", len(TEST_SAMPLES))

Dataset split information saved.
Random Seed: 42
Training: 16
Validation: 2384
Testing: 400


## 8. Experiment Results Tracking

In [8]:
RESULTS_PATH = ROOT / "experiment_results.csv"

results_df = pd.DataFrame(columns=[
    "Experiment",
    "Architecture",
    "Method",
    "Validation Accuracy (%)",
    "Test Accuracy (%)",
    "Runtime (seconds)",
    "Observations"
])


def record_experiment(
    experiment,
    architecture,
    method,
    validation_accuracy,
    test_accuracy,
    runtime_seconds,
    observations=""
):
    global results_df

    new_row = pd.DataFrame([{
        "Experiment": experiment,
        "Architecture": architecture,
        "Method": method,
        "Validation Accuracy (%)": round(validation_accuracy, 4),
        "Test Accuracy (%)": round(test_accuracy, 4),
        "Runtime (seconds)": round(runtime_seconds, 2),
        "Observations": observations
    }])

    results_df = pd.concat(
        [
            results_df[results_df["Experiment"] != experiment],
            new_row
        ],
        ignore_index=True
    )

    results_df.to_csv(RESULTS_PATH, index=False)

    display(results_df)


display(results_df)
print("Experiment tracking initialized successfully.")

,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (seconds),Observations


Experiment tracking initialized successfully.


# Experiment 1: ConvNeXt-Base (ImageNet-22K) + CLIP ViT-L/14 (QuickGELU)

**Method:** Frozen pretrained feature extraction, one-shot cosine similarity, and fixed equal-weight score fusion.

**Dataset:** One labeled reference image per class (16 images total).

**Evaluation:** Individual model performance and combined ensemble performance.


### 1.1 Model Initialization and Preprocessing

In [ ]:
%pip -q install timm open_clip_torch

import time
import json
import hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import timm
import open_clip
from pathlib import Path

E1_START = time.time()
E1_DIR = ROOT / "experiment_1"
E1_DIR.mkdir(parents=True, exist_ok=True)

E1_CONVNEXT_ID = "convnext_base.fb_in22k"
E1_CLIP_ID = "ViT-L-14-quickgelu"
E1_CLIP_WEIGHTS = "openai"

assert torch.cuda.is_available(), "Enable T4 GPU in Colab."

assert len(ONE_SHOT_SAMPLES) == 16
assert len(VALIDATION_SAMPLES) == 2384
assert len(TEST_SAMPLES) == 400

def e1_path_set(samples):
    return {str(Path(p).resolve()) for p, _ in samples}

assert e1_path_set(ONE_SHOT_SAMPLES).isdisjoint(
    e1_path_set(VALIDATION_SAMPLES)
)
assert e1_path_set(TEST_SAMPLES).isdisjoint(
    e1_path_set(ONE_SHOT_SAMPLES) |
    e1_path_set(VALIDATION_SAMPLES)
)

assert sorted(int(y) for _, y in ONE_SHOT_SAMPLES) == list(range(16))

# Cache identity includes dataset paths, labels, and model checkpoints.
split_info = [
    sorted((str(Path(p).resolve()), int(y)) for p, y in samples)
    for samples in (ONE_SHOT_SAMPLES, VALIDATION_SAMPLES, TEST_SAMPLES)
]
cache_identity = {
    "splits": split_info,
    "convnext": E1_CONVNEXT_ID,
    "clip": E1_CLIP_ID,
    "clip_weights": E1_CLIP_WEIGHTS
}
E1_CACHE_ID = hashlib.sha256(
    json.dumps(cache_identity).encode()
).hexdigest()[:16]

# Load pretrained models on CPU to avoid GPU memory overload.
convnext = timm.create_model(
    E1_CONVNEXT_ID,
    pretrained=True,
    num_classes=0
).eval()

convnext_config = timm.data.resolve_model_data_config(convnext)
convnext_transform = timm.data.create_transform(
    **convnext_config,
    is_training=False
)

clip_model, _, clip_transform = (
    open_clip.create_model_and_transforms(
        E1_CLIP_ID,
        pretrained=E1_CLIP_WEIGHTS,
        device="cpu"
    )
)
clip_model.eval()

print("EXPERIMENT 1 INITIALIZED SUCCESSFULLY")
print("ConvNeXt:", E1_CONVNEXT_ID)
print("CLIP:", E1_CLIP_ID, E1_CLIP_WEIGHTS)
print("GPU:", torch.cuda.get_device_name(0))
print("Training:", len(ONE_SHOT_SAMPLES))
print("Validation:", len(VALIDATION_SAMPLES))
print("Test:", len(TEST_SAMPLES))
print("Training/validation path overlap: None")

EXPERIMENT 1 INITIALIZED SUCCESSFULLY
ConvNeXt: convnext_base.fb_in22k
CLIP: ViT-L-14-quickgelu openai
GPU: Tesla T4
Training: 16
Validation: 2384
Test: 400
Training/validation path overlap: None


### 1.2 One-Shot Feature Extraction and Evaluation

Extracting normalized features from the 16 labeled reference images using both frozen pretrained models. Computing cosine-similarity class scores and combining standardized scores using fixed equal weights (0.50 and 0.50). No additional labeled images are used to optimize the fusion weights.

In [ ]:
def e1_extract_features(
    model, model_type, samples, transform, batch_size=16
):
    loader = create_loader(
        samples, transform, batch_size=batch_size
    )
    features, labels = [], []

    model = model.to(DEVICE).eval()

    try:
        with torch.inference_mode():
            for step, (images, targets) in enumerate(loader):
                images = images.to(DEVICE, non_blocking=True)

                with torch.autocast(
                    device_type="cuda", dtype=torch.float16
                ):
                    if model_type == "clip":
                        embeddings = model.encode_image(images)
                    else:
                        embeddings = model(images)

                embeddings = F.normalize(
                    embeddings.float(), dim=1
                )

                features.append(embeddings.cpu())
                labels.append(targets.cpu())

                if (step + 1) % 25 == 0:
                    print(
                        f"Processed {min((step + 1) * batch_size, len(samples))}"
                        f"/{len(samples)} images"
                    )
    finally:
        model.cpu()
        cleanup_gpu()

    return torch.cat(features), torch.cat(labels)


def e1_get_features(
    name, model, model_type, samples, transform
):
    cache = E1_DIR / f"{name}_{E1_CACHE_ID}.pt"

    if cache.exists():
        print("Loading cached features:", name)
        saved = torch.load(
            cache, map_location="cpu", weights_only=True
        )
        return saved["features"], saved["labels"]

    print("\nExtracting features:", name)

    features, labels = e1_extract_features(
        model, model_type, samples, transform
    )

    torch.save(
        {"features": features, "labels": labels},
        cache
    )
    return features, labels


def e1_class_scores(features, references, reference_labels):
    order = torch.argsort(reference_labels)
    assert torch.equal(
        reference_labels[order], torch.arange(16)
    )
    return features @ references[order].T


def e1_normalize_scores(scores):
    return (
        (scores - scores.mean(dim=1, keepdim=True))
        / scores.std(dim=1, keepdim=True).clamp_min(1e-6)
    )


def e1_accuracy(scores, labels):
    return (
        (scores.argmax(dim=1) == labels)
        .float().mean().item() * 100
    )


# ConvNeXt reference and validation features.
cn_ref, cn_ref_y = e1_get_features(
    "convnext_reference", convnext, "convnext",
    ONE_SHOT_SAMPLES, convnext_transform
)

cn_val, cn_val_y = e1_get_features(
    "convnext_validation", convnext, "convnext",
    VALIDATION_SAMPLES, convnext_transform
)

# CLIP reference and validation features.
clip_ref, clip_ref_y = e1_get_features(
    "clip_reference", clip_model, "clip",
    ONE_SHOT_SAMPLES, clip_transform
)

clip_val, clip_val_y = e1_get_features(
    "clip_validation", clip_model, "clip",
    VALIDATION_SAMPLES, clip_transform
)

assert torch.equal(cn_val_y, clip_val_y)

# Compute individual cosine similarity scores.
cn_val_scores = e1_class_scores(
    cn_val, cn_ref, cn_ref_y
)
clip_val_scores = e1_class_scores(
    clip_val, clip_ref, clip_ref_y
)

# Individual model validation accuracy (diagnostic).
E1_CN_VAL_ACC = e1_accuracy(cn_val_scores, cn_val_y)
E1_CLIP_VAL_ACC = e1_accuracy(clip_val_scores, clip_val_y)

print("\nINDIVIDUAL MODEL VALIDATION ANALYSIS")
print(f"ConvNeXt-Base Validation: {E1_CN_VAL_ACC:.2f}%")
print(f"CLIP ViT-L/14 Validation: {E1_CLIP_VAL_ACC:.2f}%")

# Normalize scores for weighted fusion.
cn_val_norm = e1_normalize_scores(cn_val_scores)
clip_val_norm = e1_normalize_scores(clip_val_scores)

# Predetermined equal weights: no validation-label tuning.
E1_BEST_WEIGHT = 0.50
E1_VAL_ACCURACY = e1_accuracy(
    0.50 * cn_val_norm + 0.50 * clip_val_norm,
    cn_val_y
)

print("\nCOMBINED MODEL VALIDATION (REPORTING ONLY)")
print(f"Overall Validation Accuracy: {E1_VAL_ACCURACY:.2f}%")
print("Fusion weights: 0.50 / 0.50 (fixed before evaluation)")
print("Validation labels were not used to select model parameters or weights.")

Loading cached features: convnext_reference
Loading cached features: convnext_validation
Loading cached features: clip_reference
Loading cached features: clip_validation

INDIVIDUAL MODEL VALIDATION ANALYSIS
ConvNeXt-Base Validation: 80.83%
CLIP ViT-L/14 Validation: 66.82%

COMBINED MODEL VALIDATION (REPORTING ONLY)
Overall Validation Accuracy: 80.12%
Fusion weights: 0.50 / 0.50 (fixed before evaluation)
Validation labels were not used to select model parameters or weights.


### 1.3 Test Evaluation, Score Fusion, and Results

In [ ]:
# ============================================================
# EXPERIMENT 1 — TEST EVALUATION, SCORE FUSION, AND RESULTS
# ConvNeXt-Base (ImageNet-22K) + CLIP ViT-L/14 (QuickGELU)
# ============================================================

# 1. Extract test features from both pretrained models.

cn_test, cn_test_y = e1_get_features(
    "convnext_test",
    convnext,
    "convnext",
    TEST_SAMPLES,
    convnext_transform
)

clip_test, clip_test_y = e1_get_features(
    "clip_test",
    clip_model,
    "clip",
    TEST_SAMPLES,
    clip_transform
)

assert torch.equal(cn_test_y, clip_test_y), (
    "Test labels do not match between the two models."
)
assert len(cn_test_y) == 400, "Expected exactly 400 test images."


# 2. Compute individual model cosine similarity scores.

cn_test_scores = e1_class_scores(
    cn_test,
    cn_ref,
    cn_ref_y
)

clip_test_scores = e1_class_scores(
    clip_test,
    clip_ref,
    clip_ref_y
)


# 3. Calculate individual model test accuracies.

E1_CN_TEST_ACC = e1_accuracy(
    cn_test_scores,
    cn_test_y
)

E1_CLIP_TEST_ACC = e1_accuracy(
    clip_test_scores,
    clip_test_y
)

print("\nINDIVIDUAL MODEL TEST ANALYSIS")
print(f"ConvNeXt-Base Test Accuracy: {E1_CN_TEST_ACC:.2f}%")
print(f"CLIP ViT-L/14 Test Accuracy: {E1_CLIP_TEST_ACC:.2f}%")


# 4. Apply predetermined equal fusion weights.

fused_test_scores = (
    E1_BEST_WEIGHT * e1_normalize_scores(cn_test_scores)
    + (1 - E1_BEST_WEIGHT)
    * e1_normalize_scores(clip_test_scores)
)

E1_TEST_ACCURACY = e1_accuracy(
    fused_test_scores,
    cn_test_y
)

E1_RUNTIME = time.time() - E1_START


# 5. Save individual and combined model accuracies for Excel.

e1_diagnostics = pd.DataFrame([
    {
        "Experiment": "E1",
        "Model": "ConvNeXt-Base (ImageNet-22K)",
        "Validation Accuracy (%)": round(E1_CN_VAL_ACC, 2),
        "Test Accuracy (%)": round(E1_CN_TEST_ACC, 2)
    },
    {
        "Experiment": "E1",
        "Model": "CLIP ViT-L/14 (QuickGELU)",
        "Validation Accuracy (%)": round(E1_CLIP_VAL_ACC, 2),
        "Test Accuracy (%)": round(E1_CLIP_TEST_ACC, 2)
    },
    {
        "Experiment": "E1",
        "Model": "ConvNeXt-Base + CLIP ViT-L/14 (Fusion)",
        "Validation Accuracy (%)": round(E1_VAL_ACCURACY, 2),
        "Test Accuracy (%)": round(E1_TEST_ACCURACY, 2)
    }
])

e1_diagnostics.to_csv(
    E1_DIR / "experiment_1_diagnostics.csv",
    index=False
)

print("\nMODEL COMPARISON — SAVED FOR EXCEL")
display(e1_diagnostics)


# 6. Create the final experiment results table.
# Only the overall combined model accuracies are included.

e1_results = pd.DataFrame([{
    "Experiment": "E1",
    "Architecture": (
        "ConvNeXt-Base (ImageNet-22K) + "
        "CLIP ViT-L/14 (QuickGELU)"
    ),
    "Method": "One-shot cosine similarity + fixed equal-weight score fusion",
    "Validation Accuracy (%)": round(E1_VAL_ACCURACY, 2),
    "Test Accuracy (%)": round(E1_TEST_ACCURACY, 2),
    "Runtime (minutes)": round(E1_RUNTIME / 60, 2)
}])

e1_results.to_csv(
    E1_DIR / "experiment_1_results.csv",
    index=False
)


# 7. Save predictions and configuration for reproducibility.

torch.save({
    "test_labels": cn_test_y,
    "test_predictions": fused_test_scores.argmax(dim=1),
    "convnext_weight": E1_BEST_WEIGHT,
    "clip_weight": 1 - E1_BEST_WEIGHT,
    "convnext_checkpoint": E1_CONVNEXT_ID,
    "clip_checkpoint": E1_CLIP_ID,
    "clip_pretrained_weights": E1_CLIP_WEIGHTS,
    "cache_identity": E1_CACHE_ID,
    "validation_accuracy": E1_VAL_ACCURACY,
    "test_accuracy": E1_TEST_ACCURACY
}, E1_DIR / "experiment_1_predictions.pt")


# 8. Update the common experiment tracking table.

record_experiment(
    experiment="E1",
    architecture=(
        "ConvNeXt-Base (ImageNet-22K) + "
        "CLIP ViT-L/14 (QuickGELU)"
    ),
    method="One-shot cosine similarity + fixed equal-weight score fusion",
    validation_accuracy=E1_VAL_ACCURACY,
    test_accuracy=E1_TEST_ACCURACY,
    runtime_seconds=E1_RUNTIME,
    observations=(
        f"Frozen pretrained models; no epochs; "
        f"ConvNeXt weight={E1_BEST_WEIGHT:.2f}; "
        f"CLIP weight={1-E1_BEST_WEIGHT:.2f}"
    )
)


# 9. Display only the final combined experiment results.

print("\n" + "=" * 65)
print("EXPERIMENT 1 — FINAL COMBINED RESULTS")
print("=" * 65)

display(e1_results)

print(f"\nOverall Validation Accuracy: {E1_VAL_ACCURACY:.2f}%")
print(f"Overall Test Accuracy: {E1_TEST_ACCURACY:.2f}%")
print(f"Total Runtime: {E1_RUNTIME / 60:.2f} minutes")

print("\nExperiment 1 completed successfully.")
print("All results have been saved.")


# 10. Download result files to preserve them.

from google.colab import files

files.download(str(E1_DIR / "experiment_1_results.csv"))
files.download(str(E1_DIR / "experiment_1_diagnostics.csv"))
files.download(str(ROOT / "experiment_results.csv"))


Loading cached features: convnext_test
Loading cached features: clip_test

INDIVIDUAL MODEL TEST ANALYSIS
ConvNeXt-Base Test Accuracy: 79.75%
CLIP ViT-L/14 Test Accuracy: 72.25%

MODEL COMPARISON — SAVED FOR EXCEL


,Experiment,Model,Validation Accuracy (%),Test Accuracy (%)
0,E1,ConvNeXt-Base (ImageNet-22K),80.83,79.75
1,E1,CLIP ViT-L/14 (QuickGELU),66.82,72.25
2,E1,ConvNeXt-Base + CLIP ViT-L/14 (Fusion),80.12,81.25


,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (seconds),Observations
0,E2,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + weighted score fu...,84.9413,86.25,194.55,Frozen pretrained models; no epochs; ResNet we...
1,E1,ConvNeXt-Base (ImageNet-22K) + CLIP ViT-L/14 (...,One-shot cosine similarity + fixed equal-weigh...,80.1175,81.25,34.69,Frozen pretrained models; no epochs; ConvNeXt ...



EXPERIMENT 1 — FINAL COMBINED RESULTS


,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (minutes)
0,E1,ConvNeXt-Base (ImageNet-22K) + CLIP ViT-L/14 (...,One-shot cosine similarity + fixed equal-weigh...,80.12,81.25,0.58



Overall Validation Accuracy: 80.12%
Overall Test Accuracy: 81.25%
Total Runtime: 0.58 minutes

Experiment 1 completed successfully.
All results have been saved.


... (2 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Experiment 2: ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/16, WebLI)

**Method:** Frozen pretrained feature extraction, one-shot cosine similarity, and fixed equal-weight score fusion.

**Dataset:** One labeled reference image per class (16 images total).

**Evaluation:** Individual model performance and combined ensemble performance.


### 2.1 Model Initialization and Preprocessing

In [20]:
%pip -q install -U open_clip_torch

import time
import json
import hashlib
import subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms
import open_clip
from pathlib import Path

E2_START = time.time()
E2_DIR = ROOT / "experiment_2"
E2_DIR.mkdir(parents=True, exist_ok=True)

E2_RESNET_ID = "resnet50_places365"
E2_SIGLIP_ID = "ViT-B-16-SigLIP2"
E2_SIGLIP_WEIGHTS = "webli"

assert torch.cuda.is_available(), "Enable T4 GPU in Colab."
assert len(ONE_SHOT_SAMPLES) == 16
assert len(VALIDATION_SAMPLES) == 2384
assert len(TEST_SAMPLES) == 400

def e2_path_set(samples):
    return {str(Path(p).resolve()) for p, _ in samples}

assert e2_path_set(ONE_SHOT_SAMPLES).isdisjoint(
    e2_path_set(VALIDATION_SAMPLES)
)
assert e2_path_set(TEST_SAMPLES).isdisjoint(
    e2_path_set(ONE_SHOT_SAMPLES) |
    e2_path_set(VALIDATION_SAMPLES)
)

assert sorted(int(y) for _, y in ONE_SHOT_SAMPLES) == list(range(16))

split_info = [
    sorted(
        (str(Path(p).resolve()), int(y))
        for p, y in samples
    )
    for samples in (
        ONE_SHOT_SAMPLES,
        VALIDATION_SAMPLES,
        TEST_SAMPLES
    )
]

E2_CACHE_ID = hashlib.sha256(
    json.dumps({
        "splits": split_info,
        "resnet": E2_RESNET_ID,
        "siglip": E2_SIGLIP_ID,
        "siglip_weights": E2_SIGLIP_WEIGHTS
    }).encode()
).hexdigest()[:16]

# --------------------------------------------------
# Download ResNet-50 Places365 pretrained weights.
# Same HTTP wget method used in Assignment 1.
# --------------------------------------------------

E2_RESNET_FILE = E2_DIR / "resnet50_places365.pth.tar"

E2_RESNET_URL = (
    "http://places2.csail.mit.edu/models_places365/"
    "resnet50_places365.pth.tar"
)

print("Downloading ResNet-50 Places365 weights...")

if not E2_RESNET_FILE.exists() or E2_RESNET_FILE.stat().st_size < 1_000_000:
    E2_RESNET_FILE.unlink(missing_ok=True)

    try:
        subprocess.run(
            [
                "wget",
                "-q",
                "--timeout=30",
                "--tries=2",
                E2_RESNET_URL,
                "-O",
                str(E2_RESNET_FILE)
            ],
            check=True,
            timeout=90
        )
    except Exception as exc:
        E2_RESNET_FILE.unlink(missing_ok=True)
        raise RuntimeError(
            "ResNet-50 Places365 download failed. "
            "The checkpoint server may be unavailable."
        ) from exc

print("Places365 weights downloaded.")
print(
    "Checkpoint size:",
    round(E2_RESNET_FILE.stat().st_size / (1024**2), 2),
    "MB"
)

# --------------------------------------------------
# Initialize ResNet-50 with Places365 weights.
# --------------------------------------------------

resnet = models.resnet50(
    weights=None,
    num_classes=365
)

checkpoint = torch.load(
    E2_RESNET_FILE,
    map_location="cpu",
    weights_only=False
)

state_dict = checkpoint.get("state_dict", checkpoint)

state_dict = {
    key.removeprefix("module."): value
    for key, value in state_dict.items()
}

resnet.load_state_dict(state_dict, strict=True)

# Remove classification layer for feature extraction.
resnet.fc = nn.Identity()
resnet.eval()

resnet_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("ResNet-50 Places365 initialized successfully.")

# --------------------------------------------------
# Initialize SigLIP 2 Base pretrained on WebLI.
# --------------------------------------------------

print("\nLoading SigLIP 2 Base...")

siglip_model, _, siglip_transform = (
    open_clip.create_model_and_transforms(
        E2_SIGLIP_ID,
        pretrained=E2_SIGLIP_WEIGHTS,
        device="cpu"
    )
)

siglip_model.eval()

print("SigLIP 2 Base initialized successfully.")

# --------------------------------------------------
# Final initialization summary.
# --------------------------------------------------

print("\n" + "=" * 55)
print("EXPERIMENT 2 INITIALIZED SUCCESSFULLY")
print("=" * 55)

print("ResNet:", E2_RESNET_ID)
print("SigLIP 2:", E2_SIGLIP_ID)
print("Pretrained weights:", E2_SIGLIP_WEIGHTS)
print("GPU:", torch.cuda.get_device_name(0))

print("Training images:", len(ONE_SHOT_SAMPLES))
print("Validation images:", len(VALIDATION_SAMPLES))
print("Test images:", len(TEST_SAMPLES))

print("Training/validation path overlap: None")
print("\nReady for Code Cell 2.")

Places365 weights downloaded.
Checkpoint size: 92.76 MB
ResNet-50 Places365 initialized successfully.

Loading SigLIP 2 Base...
SigLIP 2 Base initialized successfully.

EXPERIMENT 2 INITIALIZED SUCCESSFULLY
ResNet: resnet50_places365
SigLIP 2: ViT-B-16-SigLIP2
Pretrained weights: webli
GPU: Tesla T4
Training images: 16
Validation images: 2384
Test images: 400
Training/validation path overlap: None

Ready for Code Cell 2.


### 2.2 One-Shot Feature Extraction and Evaluation

Extracting normalized features using one labeled reference image per class. Calculating cosine-similarity class scores and combining standardized scores using predetermined equal weights (0.50 / 0.50).

Validation accuracy is reported for diagnostic purposes only. Validation labels are not used to select fusion weights or other model parameters.

In [21]:
def e2_extract_features(
    model, model_type, samples, transform, batch_size=16
):
    loader = create_loader(
        samples, transform, batch_size=batch_size
    )
    features, labels = [], []
    model = model.to(DEVICE).eval()

    try:
        with torch.inference_mode():
            for step, (images, targets) in enumerate(loader):
                images = images.to(DEVICE, non_blocking=True)

                with torch.autocast(
                    device_type="cuda", dtype=torch.float16
                ):
                    if model_type == "siglip":
                        embeddings = model.encode_image(images)
                    else:
                        embeddings = model(images)

                embeddings = F.normalize(
                    embeddings.float(), dim=1
                )

                features.append(embeddings.cpu())
                labels.append(targets.cpu())

                if (step + 1) % 25 == 0:
                    print(
                        f"Processed "
                        f"{min((step + 1) * batch_size, len(samples))}"
                        f"/{len(samples)} images"
                    )
    finally:
        model.cpu()
        cleanup_gpu()

    return torch.cat(features), torch.cat(labels)


def e2_get_features(
    name, model, model_type, samples, transform
):
    cache = E2_DIR / f"{name}_{E2_CACHE_ID}.pt"

    if cache.exists():
        print("Loading cached features:", name)
        saved = torch.load(
            cache, map_location="cpu", weights_only=True
        )
        return saved["features"], saved["labels"]

    print("\nExtracting features:", name)

    features, labels = e2_extract_features(
        model, model_type, samples, transform
    )

    torch.save(
        {"features": features, "labels": labels},
        cache
    )
    return features, labels


def e2_class_scores(features, references, reference_labels):
    order = torch.argsort(reference_labels)

    assert torch.equal(
        reference_labels[order], torch.arange(16)
    )

    return features @ references[order].T


def e2_normalize_scores(scores):
    return (
        (scores - scores.mean(dim=1, keepdim=True))
        / scores.std(dim=1, keepdim=True).clamp_min(1e-6)
    )


def e2_accuracy(scores, labels):
    return (
        (scores.argmax(dim=1) == labels)
        .float().mean().item() * 100
    )


# ResNet-50 reference and validation features.
rn_ref, rn_ref_y = e2_get_features(
    "resnet_reference", resnet, "resnet",
    ONE_SHOT_SAMPLES, resnet_transform
)

rn_val, rn_val_y = e2_get_features(
    "resnet_validation", resnet, "resnet",
    VALIDATION_SAMPLES, resnet_transform
)

# SigLIP 2 reference and validation features.
sig_ref, sig_ref_y = e2_get_features(
    "siglip_reference", siglip_model, "siglip",
    ONE_SHOT_SAMPLES, siglip_transform
)

sig_val, sig_val_y = e2_get_features(
    "siglip_validation", siglip_model, "siglip",
    VALIDATION_SAMPLES, siglip_transform
)

assert torch.equal(rn_val_y, sig_val_y)
assert len(rn_val_y) == 2384

# Individual cosine similarity scores.
rn_val_scores = e2_class_scores(
    rn_val, rn_ref, rn_ref_y
)

sig_val_scores = e2_class_scores(
    sig_val, sig_ref, sig_ref_y
)

# Individual validation accuracy for diagnostic analysis.
E2_RN_VAL_ACC = e2_accuracy(rn_val_scores, rn_val_y)
E2_SIG_VAL_ACC = e2_accuracy(sig_val_scores, rn_val_y)

print("\nINDIVIDUAL MODEL VALIDATION ANALYSIS")
print(f"ResNet-50 Validation: {E2_RN_VAL_ACC:.2f}%")
print(f"SigLIP 2 Base Validation: {E2_SIG_VAL_ACC:.2f}%")

# Normalize class scores before fusion.
rn_val_norm = e2_normalize_scores(rn_val_scores)
sig_val_norm = e2_normalize_scores(sig_val_scores)

# Predetermined equal weights: no validation-label tuning.
E2_BEST_WEIGHT = 0.50
E2_VAL_ACCURACY = e2_accuracy(
    0.50 * rn_val_norm + 0.50 * sig_val_norm,
    rn_val_y
)

print("\nCOMBINED MODEL VALIDATION (REPORTING ONLY)")
print(f"Overall Validation Accuracy: {E2_VAL_ACCURACY:.2f}%")
print("Fusion weights: 0.50 / 0.50 (fixed before evaluation)")
print("Validation labels were not used to select model parameters or weights.")


Extracting features: resnet_reference

Extracting features: resnet_validation
Processed 400/2384 images
Processed 800/2384 images
Processed 1200/2384 images
Processed 1600/2384 images
Processed 2000/2384 images

Extracting features: siglip_reference

Extracting features: siglip_validation
Processed 400/2384 images
Processed 800/2384 images
Processed 1200/2384 images
Processed 1600/2384 images
Processed 2000/2384 images

INDIVIDUAL MODEL VALIDATION ANALYSIS
ResNet-50 Validation: 81.67%
SigLIP 2 Base Validation: 78.61%

COMBINED MODEL VALIDATION (REPORTING ONLY)
Overall Validation Accuracy: 84.10%
Fusion weights: 0.50 / 0.50 (fixed before evaluation)
Validation labels were not used to select model parameters or weights.


### 2.3 Test Evaluation, Score Fusion, and Results

In [22]:
# 1. Extract test features.
rn_test, rn_test_y = e2_get_features(
    "resnet_test", resnet, "resnet",
    TEST_SAMPLES, resnet_transform
)

sig_test, sig_test_y = e2_get_features(
    "siglip_test", siglip_model, "siglip",
    TEST_SAMPLES, siglip_transform
)

assert torch.equal(rn_test_y, sig_test_y)
assert len(rn_test_y) == 400

# 2. Calculate individual cosine similarity scores.
rn_test_scores = e2_class_scores(
    rn_test, rn_ref, rn_ref_y
)

sig_test_scores = e2_class_scores(
    sig_test, sig_ref, sig_ref_y
)

# 3. Individual test accuracies for diagnostic analysis.
E2_RN_TEST_ACC = e2_accuracy(
    rn_test_scores, rn_test_y
)

E2_SIG_TEST_ACC = e2_accuracy(
    sig_test_scores, sig_test_y
)

print("\nINDIVIDUAL MODEL TEST ANALYSIS")
print(f"ResNet-50 Test Accuracy: {E2_RN_TEST_ACC:.2f}%")
print(f"SigLIP 2 Base Test Accuracy: {E2_SIG_TEST_ACC:.2f}%")

# 4. Apply predetermined equal fusion weights.
fused_test_scores = (
    E2_BEST_WEIGHT * e2_normalize_scores(rn_test_scores)
    + (1 - E2_BEST_WEIGHT)
    * e2_normalize_scores(sig_test_scores)
)

E2_TEST_ACCURACY = e2_accuracy(
    fused_test_scores, rn_test_y
)

E2_RUNTIME = time.time() - E2_START

# 5. Save individual and combined accuracies for Excel.
e2_diagnostics = pd.DataFrame([
    {
        "Experiment": "E2",
        "Model": "ResNet-50 (Places365)",
        "Validation Accuracy (%)": round(E2_RN_VAL_ACC, 2),
        "Test Accuracy (%)": round(E2_RN_TEST_ACC, 2)
    },
    {
        "Experiment": "E2",
        "Model": "SigLIP 2 Base (ViT-B/16, 224)",
        "Validation Accuracy (%)": round(E2_SIG_VAL_ACC, 2),
        "Test Accuracy (%)": round(E2_SIG_TEST_ACC, 2)
    },
    {
        "Experiment": "E2",
        "Model": "ResNet-50 + SigLIP 2 (Fusion)",
        "Validation Accuracy (%)": round(E2_VAL_ACCURACY, 2),
        "Test Accuracy (%)": round(E2_TEST_ACCURACY, 2)
    }
])

e2_diagnostics.to_csv(
    E2_DIR / "experiment_2_diagnostics.csv",
    index=False
)

print("\nMODEL COMPARISON — SAVED FOR EXCEL")
display(e2_diagnostics)

# 6. Create the final combined experiment results.
e2_results = pd.DataFrame([{
    "Experiment": "E2",
    "Architecture": (
        "ResNet-50 (Places365) + "
        "SigLIP 2 Base (ViT-B/16, 224)"
    ),
    "Method": "One-shot cosine similarity + fixed equal-weight score fusion",
    "Validation Accuracy (%)": round(E2_VAL_ACCURACY, 2),
    "Test Accuracy (%)": round(E2_TEST_ACCURACY, 2),
    "Runtime (minutes)": round(E2_RUNTIME / 60, 2)
}])

e2_results.to_csv(
    E2_DIR / "experiment_2_results.csv",
    index=False
)

# 7. Save predictions and configuration.
torch.save({
    "test_labels": rn_test_y,
    "test_predictions": fused_test_scores.argmax(dim=1),
    "resnet_weight": E2_BEST_WEIGHT,
    "siglip_weight": 1 - E2_BEST_WEIGHT,
    "resnet_checkpoint": E2_RESNET_ID,
    "siglip_checkpoint": E2_SIGLIP_ID,
    "siglip_pretrained_weights": E2_SIGLIP_WEIGHTS,
    "cache_identity": E2_CACHE_ID,
    "validation_accuracy": E2_VAL_ACCURACY,
    "test_accuracy": E2_TEST_ACCURACY
}, E2_DIR / "experiment_2_predictions.pt")

# 8. Update the common experiment tracking table.
record_experiment(
    experiment="E2",
    architecture=(
        "ResNet-50 (Places365) + "
        "SigLIP 2 Base (ViT-B/16, 224)"
    ),
    method="One-shot cosine similarity + fixed equal-weight score fusion",
    validation_accuracy=E2_VAL_ACCURACY,
    test_accuracy=E2_TEST_ACCURACY,
    runtime_seconds=E2_RUNTIME,
    observations=(
        f"Frozen pretrained models; no epochs; "
        f"ResNet weight={E2_BEST_WEIGHT:.2f}; "
        f"SigLIP weight={1-E2_BEST_WEIGHT:.2f}"
    )
)

# 9. Display only the final combined results.
print("\n" + "=" * 65)
print("EXPERIMENT 2 — FINAL COMBINED RESULTS")
print("=" * 65)

display(e2_results)

print(f"\nOverall Validation Accuracy: {E2_VAL_ACCURACY:.2f}%")
print(f"Overall Test Accuracy: {E2_TEST_ACCURACY:.2f}%")
print(f"Total Runtime: {E2_RUNTIME / 60:.2f} minutes")

print("\nExperiment 2 completed successfully.")
print("All results have been saved.")

# 10. Download results to your computer.
from google.colab import files

files.download(str(E2_DIR / "experiment_2_results.csv"))
files.download(str(E2_DIR / "experiment_2_diagnostics.csv"))
files.download(str(ROOT / "experiment_results.csv"))


Extracting features: resnet_test
Processed 400/400 images

Extracting features: siglip_test
Processed 400/400 images

INDIVIDUAL MODEL TEST ANALYSIS
ResNet-50 Test Accuracy: 83.50%
SigLIP 2 Base Test Accuracy: 78.75%

MODEL COMPARISON — SAVED FOR EXCEL


,Experiment,Model,Validation Accuracy (%),Test Accuracy (%)
0,E2,ResNet-50 (Places365),81.67,83.50
1,E2,"SigLIP 2 Base (ViT-B/16, 224)",78.61,78.75
2,E2,ResNet-50 + SigLIP 2 (Fusion),84.10,85.25


,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (seconds),Observations
0,E4,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + five-view center-...,82.2987,82.75,288.21,Three frozen pretrained models; five determini...
1,E5,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + reference-only pr...,84.1862,85.00,86.35,Two frozen pretrained models; mild reference-o...
2,E2,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + fixed equal-weigh...,84.1024,85.25,74.60,Frozen pretrained models; no epochs; ResNet we...



EXPERIMENT 2 — FINAL COMBINED RESULTS


,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (minutes)
0,E2,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + fixed equal-weigh...,84.1,85.25,1.24



Overall Validation Accuracy: 84.10%
Overall Test Accuracy: 85.25%
Total Runtime: 1.24 minutes

Experiment 2 completed successfully.
All results have been saved.


... (2 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Experiment 3: ResNet-50 (Places365) + SigLIP 2 Base (WebLI) + DINOv2 ViT-B/14 (LVD-142M)

**Objective:** Investigate whether incorporating self-supervised DINOv2 visual representations improves the one-shot classification performance of the ResNet-50 and SigLIP 2 ensemble evaluated in Experiment 2.

**Architecture:**
- ResNet-50 pretrained on Places365.
- SigLIP 2 Base (ViT-B/16) pretrained on WebLI.
- DINOv2 ViT-B/14 pretrained on LVD-142M.

**Method:** Frozen pretrained feature extraction, one-shot cosine similarity, per-image score standardization, and fixed equal-weight three-model fusion.

**One-shot restriction:** Exactly one labeled reference image per class is used for classification. No additional labeled images are used for model training, fine-tuning, or fusion-weight optimization.

**Fusion weights:** ResNet-50 = 1/3, SigLIP 2 = 1/3, and DINOv2 = 1/3.

**Evaluation:** Individual model accuracies and combined ensemble accuracy are reported on the validation and test datasets. Validation labels are used only for diagnostic evaluation, not for selecting model parameters or fusion weights.

**Baseline:** Experiment 2 achieved 85.25% test accuracy using ResNet-50 and SigLIP 2 with fixed 50/50 score fusion.

**Implementation:** All three pretrained models remain frozen. Extracted embeddings are cached to reduce repeated computation and GPU usage.

### 3.1 Model Initialization and Preprocessing

Initializing three pretrained visual foundation models: ResNet-50 with Places365 weights, SigLIP 2 Base with WebLI weights, and DINOv2 ViT-B/14 with LVD-142M weights.

Loading DINOv2 using the official Meta PyTorch Hub repository without requiring Hugging Face authentication.

Applying model-specific image preprocessing and keep all pretrained parameters frozen. The models operate in evaluation mode without gradient computation.

Verifying the one-shot reference set, validation set, and test set. Creating experiment-specific cache identifiers to ensure that Experiment 3 runs independently of Experiments 1 and 2.

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade",
    "transformers==4.57.3",
    "huggingface-hub==0.36.2",
    "open_clip_torch"
])

print("Installation complete.")
print("Restart the Colab session before importing transformers.")

Installation complete.
Restart the Colab session before importing transformers.


In [ ]:
# ============================================================
# EXPERIMENT 3 — MODEL INITIALIZATION AND PREPROCESSING
# ResNet-50 Places365 + SigLIP 2 Base + DINOv2 ViT-B/14
# ============================================================

import os
import gc
import json
import time
import hashlib
import subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from torchvision import transforms
from pathlib import Path
from PIL import Image
from IPython.display import display

import open_clip

E3_START = time.time()

E3_DIR = ROOT / "experiment_3"
E3_DIR.mkdir(parents=True, exist_ok=True)

E3_RESNET_ID = "resnet50_places365"
E3_SIGLIP_ID = "ViT-B-16-SigLIP2"
E3_SIGLIP_WEIGHTS = "webli"
E3_DINO_ID = "dinov2_vitb14"

E3_RESNET_URL = (
    "http://places2.csail.mit.edu/models_places365/"
    "resnet50_places365.pth.tar"
)

E3_RESNET_FILE = E3_DIR / "resnet50_places365.pth.tar"

# ------------------------------------------------------------
# 1. Verifying GPU and dataset configuration.
# ------------------------------------------------------------

assert torch.cuda.is_available(), (
    "Enable a GPU in Colab before running Experiment 3."
)

assert len(CLASS_NAMES) == 16
assert len(ONE_SHOT_SAMPLES) == 16
assert len(VALIDATION_SAMPLES) == 2384
assert len(TEST_SAMPLES) == 400

assert sorted(
    int(label) for _, label in ONE_SHOT_SAMPLES
) == list(range(16))

def e3_path_set(samples):
    return {
        str(Path(path).resolve())
        for path, _ in samples
    }

assert e3_path_set(ONE_SHOT_SAMPLES).isdisjoint(
    e3_path_set(VALIDATION_SAMPLES)
)

assert e3_path_set(TEST_SAMPLES).isdisjoint(
    e3_path_set(ONE_SHOT_SAMPLES)
    | e3_path_set(VALIDATION_SAMPLES)
)

# ------------------------------------------------------------
# 2. Generating experiment-specific cache identity.
# ------------------------------------------------------------

e3_split_info = [
    sorted(
        (str(Path(path).resolve()), int(label))
        for path, label in samples
    )
    for samples in (
        ONE_SHOT_SAMPLES,
        VALIDATION_SAMPLES,
        TEST_SAMPLES
    )
]

E3_CACHE_ID = hashlib.sha256(
    json.dumps({
        "splits": e3_split_info,
        "resnet": E3_RESNET_ID,
        "siglip": E3_SIGLIP_ID,
        "siglip_weights": E3_SIGLIP_WEIGHTS,
        "dino": E3_DINO_ID,
        "dino_feature": "normalized_cls_embedding",
        "preprocessing_version": "e3_dinov2_v1"
    }, sort_keys=True).encode()
).hexdigest()[:16]

# ------------------------------------------------------------
# 3. Initializing ResNet-50 Places365.
# ------------------------------------------------------------

print("Initializing ResNet-50 Places365...")

if (
    not E3_RESNET_FILE.exists()
    or E3_RESNET_FILE.stat().st_size < 1_000_000
):
    E3_RESNET_FILE.unlink(missing_ok=True)

    try:
        subprocess.run(
            [
                "wget",
                "-q",
                "--timeout=30",
                "--tries=2",
                E3_RESNET_URL,
                "-O",
                str(E3_RESNET_FILE)
            ],
            check=True,
            timeout=90
        )
    except Exception as exc:
        E3_RESNET_FILE.unlink(missing_ok=True)
        raise RuntimeError(
            "Places365 checkpoint download failed."
        ) from exc

e3_resnet = models.resnet50(
    weights=None,
    num_classes=365
)

e3_checkpoint = torch.load(
    E3_RESNET_FILE,
    map_location="cpu",
    weights_only=False
)

e3_state_dict = e3_checkpoint.get(
    "state_dict",
    e3_checkpoint
)

e3_state_dict = {
    key.removeprefix("module."): value
    for key, value in e3_state_dict.items()
}

e3_resnet.load_state_dict(
    e3_state_dict,
    strict=True
)

e3_resnet.fc = nn.Identity()
e3_resnet.eval()
e3_resnet.requires_grad_(False)

e3_resnet_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

del e3_checkpoint, e3_state_dict

print("ResNet-50 Places365 ready.")

# ------------------------------------------------------------
# 4. Initializing SigLIP 2 Base pretrained on WebLI.
# ------------------------------------------------------------

print("\nInitializing SigLIP 2 Base...")

e3_siglip, _, e3_siglip_transform = (
    open_clip.create_model_and_transforms(
        E3_SIGLIP_ID,
        pretrained=E3_SIGLIP_WEIGHTS,
        device="cpu"
    )
)

e3_siglip.eval()
e3_siglip.requires_grad_(False)

print("SigLIP 2 Base ready.")

# ------------------------------------------------------------
# 5. Initializing DINOv2 ViT-B/14 using PyTorch Hub.
# No Hugging Face account or token is required.
# ------------------------------------------------------------

print("\nInitializing DINOv2 ViT-B/14...")

try:
    e3_dino = torch.hub.load(
        "facebookresearch/dinov2",
        E3_DINO_ID,
        pretrained=True,
        trust_repo=True
    )
except Exception as exc:
    raise RuntimeError(
        "DINOv2 could not be downloaded from Meta's "
        "PyTorch Hub repository. Check internet access "
        "and the full error message."
    ) from exc

e3_dino.eval()
e3_dino.requires_grad_(False)

# Standard ImageNet normalization for DINOv2.
# Resize to 256, center crop to 224.
# 224 is divisible by DINOv2's patch size of 14.

e3_dino_transform = transforms.Compose([
    transforms.Resize(
        256,
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("DINOv2 ViT-B/14 ready.")

# ------------------------------------------------------------
# 6. Verifying frozen parameters.
# ------------------------------------------------------------

for model in (
    e3_resnet,
    e3_siglip,
    e3_dino
):
    assert not any(
        parameter.requires_grad
        for parameter in model.parameters()
    )

# ------------------------------------------------------------
# 7. Displaying experiment configuration.
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXPERIMENT 3 — INITIALIZATION SUCCESSFUL")
print("=" * 65)

print("ResNet:", E3_RESNET_ID)
print("SigLIP:", E3_SIGLIP_ID, E3_SIGLIP_WEIGHTS)
print("DINOv2:", E3_DINO_ID)

print("Device:", DEVICE)
print("One-shot reference images:", len(ONE_SHOT_SAMPLES))
print("Validation images:", len(VALIDATION_SAMPLES))
print("Test images:", len(TEST_SAMPLES))

print("All model parameters frozen: Yes")
print("Training epochs: 0")
print("Fusion weights: 1/3 + 1/3 + 1/3")
print("Cache identity:", E3_CACHE_ID)

print("\nReady for Experiment 3 Code Cell 2.")

Initializing ResNet-50 Places365...
ResNet-50 Places365 ready.

Initializing SigLIP 2 Base...
SigLIP 2 Base ready.

Initializing DINOv2 ViT-B/14...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:05<00:00, 61.6MB/s]


DINOv2 ViT-B/14 ready.

EXPERIMENT 3 — INITIALIZATION SUCCESSFUL
ResNet: resnet50_places365
SigLIP: ViT-B-16-SigLIP2 webli
DINOv2: dinov2_vitb14
Device: cuda
One-shot reference images: 16
Validation images: 2384
Test images: 400
All model parameters frozen: Yes
Training epochs: 0
Fusion weights: 1/3 + 1/3 + 1/3
Cache identity: dedd0ffe8ebdcc14

Ready for Experiment 3 Code Cell 2.


### 3.2 One-Shot Feature Extraction and Validation Evaluation

Extracting visual embeddings from ResNet-50, SigLIP 2, and DINOv2 for the 16 one-shot reference images and the validation images.

For each model, applying L2 normalization to the extracted embeddings and compute cosine similarity between each validation image and the 16 class reference embeddings.

Standardizing the similarity scores independently for each model and each image. Combining the standardized scores using fixed fusion weights of exactly 1/3 for ResNet-50, 1/3 for SigLIP 2, and 1/3 for DINOv2.

Recording individual validation accuracies for diagnostic comparison and calculating the combined ensemble validation accuracy. No validation-based optimization of model parameters or fusion weights is performed.

Cache the extracted embeddings separately for each model and dataset split to minimize repeated GPU computation.


In [ ]:
# ============================================================
# EXPERIMENT 3 — FEATURE EXTRACTION AND VALIDATION
# Fixed fusion: ResNet 1/3 + SigLIP 1/3 + DINOv2 1/3
# ============================================================

def e3_extract_features(
    model,
    model_type,
    samples,
    transform,
    batch_size=16
):
    loader = create_loader(
        samples,
        transform,
        batch_size=batch_size
    )

    features = []
    labels = []

    model = model.to(DEVICE).eval()

    try:
        with torch.inference_mode():
            for step, (images, targets) in enumerate(loader):

                images = images.to(
                    DEVICE,
                    non_blocking=True
                )

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):

                    if model_type == "siglip":
                        embeddings = model.encode_image(images)

                    elif model_type == "dino":
                        # Official DINOv2 model returns
                        # the global image representation.
                        embeddings = model(images)

                    elif model_type == "resnet":
                        embeddings = model(images)

                    else:
                        raise ValueError(
                            f"Unknown model type: {model_type}"
                        )

                embeddings = F.normalize(
                    embeddings.float(),
                    p=2,
                    dim=1
                )

                features.append(embeddings.cpu())
                labels.append(targets.cpu())

                if (step + 1) % 25 == 0:
                    processed = min(
                        (step + 1) * batch_size,
                        len(samples)
                    )

                    print(
                        f"{model_type}: "
                        f"{processed}/{len(samples)} images"
                    )

    finally:
        model.cpu()
        cleanup_gpu()

    return (
        torch.cat(features, dim=0),
        torch.cat(labels, dim=0)
    )


def e3_get_features(
    name,
    model,
    model_type,
    samples,
    transform
):
    cache_path = (
        E3_DIR / f"{name}_{E3_CACHE_ID}.pt"
    )

    expected_labels = torch.tensor(
        [int(label) for _, label in samples],
        dtype=torch.long
    )

    if cache_path.exists():
        try:
            saved = torch.load(
                cache_path,
                map_location="cpu",
                weights_only=True
            )

            cached_features = saved["features"]
            cached_labels = saved["labels"]

            if (
                cached_features.shape[0] == len(samples)
                and torch.equal(
                    cached_labels,
                    expected_labels
                )
                and torch.isfinite(
                    cached_features
                ).all()
            ):
                print("Loading cached features:", name)

                return (
                    cached_features,
                    cached_labels
                )

            print("Invalid cache, recomputing:", name)

        except Exception:
            print("Unreadable cache, recomputing:", name)

    print("\nExtracting features:", name)

    features, labels = e3_extract_features(
        model,
        model_type,
        samples,
        transform
    )

    assert torch.equal(
        labels,
        expected_labels
    )

    assert torch.isfinite(features).all()

    torch.save(
        {
            "features": features,
            "labels": labels
        },
        cache_path
    )

    print("Saved feature cache:", name)

    return features, labels


def e3_class_scores(
    features,
    references,
    reference_labels
):
    order = torch.argsort(reference_labels)

    assert torch.equal(
        reference_labels[order],
        torch.arange(len(CLASS_NAMES))
    ), (
        "Exactly one reference image per class "
        "is required."
    )

    reference_matrix = references[order]

    # All embeddings are L2-normalized.
    # Their dot product equals cosine similarity.

    return features @ reference_matrix.T


def e3_normalize_scores(scores):
    mean = scores.mean(
        dim=1,
        keepdim=True
    )

    std = scores.std(
        dim=1,
        keepdim=True,
        unbiased=False
    ).clamp_min(1e-6)

    return (scores - mean) / std


def e3_accuracy(scores, labels):
    predictions = scores.argmax(dim=1)

    return (
        (predictions == labels)
        .float()
        .mean()
        .item() * 100
    )


# ------------------------------------------------------------
# 1. Extracting one-shot reference embeddings.
# ------------------------------------------------------------

print("\nEXTRACTING ONE-SHOT REFERENCE FEATURES")

e3_rn_ref, e3_rn_ref_y = e3_get_features(
    "resnet_reference",
    e3_resnet,
    "resnet",
    ONE_SHOT_SAMPLES,
    e3_resnet_transform
)

e3_sig_ref, e3_sig_ref_y = e3_get_features(
    "siglip_reference",
    e3_siglip,
    "siglip",
    ONE_SHOT_SAMPLES,
    e3_siglip_transform
)

e3_dino_ref, e3_dino_ref_y = e3_get_features(
    "dinov2_reference",
    e3_dino,
    "dino",
    ONE_SHOT_SAMPLES,
    e3_dino_transform
)

assert torch.equal(
    e3_rn_ref_y,
    e3_sig_ref_y
)

assert torch.equal(
    e3_rn_ref_y,
    e3_dino_ref_y
)

# ------------------------------------------------------------
# 2. Extracting validation embeddings.
# ------------------------------------------------------------

print("\nEXTRACTING VALIDATION FEATURES")

e3_rn_val, e3_rn_val_y = e3_get_features(
    "resnet_validation",
    e3_resnet,
    "resnet",
    VALIDATION_SAMPLES,
    e3_resnet_transform
)

e3_sig_val, e3_sig_val_y = e3_get_features(
    "siglip_validation",
    e3_siglip,
    "siglip",
    VALIDATION_SAMPLES,
    e3_siglip_transform
)

e3_dino_val, e3_dino_val_y = e3_get_features(
    "dinov2_validation",
    e3_dino,
    "dino",
    VALIDATION_SAMPLES,
    e3_dino_transform
)

assert torch.equal(
    e3_rn_val_y,
    e3_sig_val_y
)

assert torch.equal(
    e3_rn_val_y,
    e3_dino_val_y
)

assert len(e3_rn_val_y) == 2384

# ------------------------------------------------------------
# 3. Calculating one-shot cosine similarity scores.
# ------------------------------------------------------------

e3_rn_val_scores = e3_class_scores(
    e3_rn_val,
    e3_rn_ref,
    e3_rn_ref_y
)

e3_sig_val_scores = e3_class_scores(
    e3_sig_val,
    e3_sig_ref,
    e3_sig_ref_y
)

e3_dino_val_scores = e3_class_scores(
    e3_dino_val,
    e3_dino_ref,
    e3_dino_ref_y
)

# ------------------------------------------------------------
# 4. Calculating individual validation accuracies.
# Diagnostic reporting only.
# ------------------------------------------------------------

E3_RN_VAL_ACC = e3_accuracy(
    e3_rn_val_scores,
    e3_rn_val_y
)

E3_SIG_VAL_ACC = e3_accuracy(
    e3_sig_val_scores,
    e3_sig_val_y
)

E3_DINO_VAL_ACC = e3_accuracy(
    e3_dino_val_scores,
    e3_dino_val_y
)

print("\nINDIVIDUAL MODEL VALIDATION ANALYSIS")

print(f"ResNet-50: {E3_RN_VAL_ACC:.2f}%")
print(f"SigLIP 2: {E3_SIG_VAL_ACC:.2f}%")
print(f"DINOv2: {E3_DINO_VAL_ACC:.2f}%")

# ------------------------------------------------------------
# 5. Fixed equal-weight three-model fusion.
# No validation-based weight optimization.
# ------------------------------------------------------------

E3_RN_WEIGHT = 1.0 / 3.0
E3_SIG_WEIGHT = 1.0 / 3.0
E3_DINO_WEIGHT = 1.0 / 3.0

assert np.isclose(
    E3_RN_WEIGHT
    + E3_SIG_WEIGHT
    + E3_DINO_WEIGHT,
    1.0
)

e3_fused_val_scores = (
    E3_RN_WEIGHT
    * e3_normalize_scores(e3_rn_val_scores)
    +
    E3_SIG_WEIGHT
    * e3_normalize_scores(e3_sig_val_scores)
    +
    E3_DINO_WEIGHT
    * e3_normalize_scores(e3_dino_val_scores)
)

E3_VAL_ACCURACY = e3_accuracy(
    e3_fused_val_scores,
    e3_rn_val_y
)

# ------------------------------------------------------------
# 6. Displaying validation results.
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("EXPERIMENT 3 — VALIDATION RESULTS")
print("=" * 60)

print(f"ResNet-50: {E3_RN_VAL_ACC:.2f}%")
print(f"SigLIP 2: {E3_SIG_VAL_ACC:.2f}%")
print(f"DINOv2: {E3_DINO_VAL_ACC:.2f}%")

print(
    f"\nOverall Validation Accuracy: "
    f"{E3_VAL_ACCURACY:.2f}%"
)

print("Fusion weights: 1/3 + 1/3 + 1/3")
print("No validation-based weight optimization performed.")

print("\nReady for Experiment 3 Code Cell 3.")


EXTRACTING ONE-SHOT REFERENCE FEATURES

Extracting features: resnet_reference
Saved feature cache: resnet_reference

Extracting features: siglip_reference
Saved feature cache: siglip_reference

Extracting features: dinov2_reference
Saved feature cache: dinov2_reference

EXTRACTING VALIDATION FEATURES

Extracting features: resnet_validation
resnet: 400/2384 images
resnet: 800/2384 images
resnet: 1200/2384 images
resnet: 1600/2384 images
resnet: 2000/2384 images
Saved feature cache: resnet_validation

Extracting features: siglip_validation
siglip: 400/2384 images
siglip: 800/2384 images
siglip: 1200/2384 images
siglip: 1600/2384 images
siglip: 2000/2384 images
Saved feature cache: siglip_validation

Extracting features: dinov2_validation
dino: 400/2384 images
dino: 800/2384 images
dino: 1200/2384 images
dino: 1600/2384 images
dino: 2000/2384 images
Saved feature cache: dinov2_validation

INDIVIDUAL MODEL VALIDATION ANALYSIS
ResNet-50: 81.67%
SigLIP 2: 78.61%
DINOv2: 75.29%

EXPERIMENT 3

### 3.3 Test Evaluation, Score Fusion, and Results

Evaluating all three frozen foundation models on the separate 400-image test set.

Calculating cosine-similarity scores using the same 16 one-shot reference embeddings. Applying the same per-model score standardization and predetermined equal fusion weights used during validation.

Reporting individual model accuracies for diagnostic analysis and the combined ensemble accuracy as the primary Experiment 3 result.

Saving model diagnostics, the combined experiment results, test predictions, fusion configuration, and reproducibility information.

The final experiment comparison table will include only the combined validation and test accuracies. No fusion weights or model parameters are adjusted using test labels.

In [ ]:
# ============================================================
# EXPERIMENT 3 — TEST EVALUATION AND FINAL RESULTS
# ResNet-50 + SigLIP 2 + DINOv2 ViT-B/14
# Fixed equal-weight three-model fusion
# ============================================================

# ------------------------------------------------------------
# 1. Extracting test embeddings.
# ------------------------------------------------------------

print("\nEXTRACTING TEST FEATURES")

e3_rn_test, e3_rn_test_y = e3_get_features(
    "resnet_test",
    e3_resnet,
    "resnet",
    TEST_SAMPLES,
    e3_resnet_transform
)

e3_sig_test, e3_sig_test_y = e3_get_features(
    "siglip_test",
    e3_siglip,
    "siglip",
    TEST_SAMPLES,
    e3_siglip_transform
)

e3_dino_test, e3_dino_test_y = e3_get_features(
    "dinov2_test",
    e3_dino,
    "dino",
    TEST_SAMPLES,
    e3_dino_transform
)

assert torch.equal(
    e3_rn_test_y,
    e3_sig_test_y
)

assert torch.equal(
    e3_rn_test_y,
    e3_dino_test_y
)

assert len(e3_rn_test_y) == 400

# ------------------------------------------------------------
# 2. Computing cosine similarity scores.
# ------------------------------------------------------------

e3_rn_test_scores = e3_class_scores(
    e3_rn_test,
    e3_rn_ref,
    e3_rn_ref_y
)

e3_sig_test_scores = e3_class_scores(
    e3_sig_test,
    e3_sig_ref,
    e3_sig_ref_y
)

e3_dino_test_scores = e3_class_scores(
    e3_dino_test,
    e3_dino_ref,
    e3_dino_ref_y
)

# ------------------------------------------------------------
# 3. Individual test accuracies.
# ------------------------------------------------------------

E3_RN_TEST_ACC = e3_accuracy(
    e3_rn_test_scores,
    e3_rn_test_y
)

E3_SIG_TEST_ACC = e3_accuracy(
    e3_sig_test_scores,
    e3_sig_test_y
)

E3_DINO_TEST_ACC = e3_accuracy(
    e3_dino_test_scores,
    e3_dino_test_y
)

print("\nINDIVIDUAL MODEL TEST ANALYSIS")

print(f"ResNet-50: {E3_RN_TEST_ACC:.2f}%")
print(f"SigLIP 2: {E3_SIG_TEST_ACC:.2f}%")
print(f"DINOv2: {E3_DINO_TEST_ACC:.2f}%")

# ------------------------------------------------------------
# 4. Applying fixed equal-weight fusion.
# ------------------------------------------------------------

e3_fused_test_scores = (
    E3_RN_WEIGHT
    * e3_normalize_scores(e3_rn_test_scores)
    +
    E3_SIG_WEIGHT
    * e3_normalize_scores(e3_sig_test_scores)
    +
    E3_DINO_WEIGHT
    * e3_normalize_scores(e3_dino_test_scores)
)

e3_test_predictions = e3_fused_test_scores.argmax(
    dim=1
)

E3_TEST_ACCURACY = e3_accuracy(
    e3_fused_test_scores,
    e3_rn_test_y
)

E3_RUNTIME = time.time() - E3_START

# ------------------------------------------------------------
# 5. Saving individual model diagnostics.
# ------------------------------------------------------------

e3_diagnostics = pd.DataFrame([
    {
        "Experiment": "E3",
        "Model": "ResNet-50 (Places365)",
        "Validation Accuracy (%)": round(E3_RN_VAL_ACC, 2),
        "Test Accuracy (%)": round(E3_RN_TEST_ACC, 2)
    },
    {
        "Experiment": "E3",
        "Model": "SigLIP 2 Base (ViT-B/16, WebLI)",
        "Validation Accuracy (%)": round(E3_SIG_VAL_ACC, 2),
        "Test Accuracy (%)": round(E3_SIG_TEST_ACC, 2)
    },
    {
        "Experiment": "E3",
        "Model": "DINOv2 ViT-B/14 (LVD-142M)",
        "Validation Accuracy (%)": round(E3_DINO_VAL_ACC, 2),
        "Test Accuracy (%)": round(E3_DINO_TEST_ACC, 2)
    },
    {
        "Experiment": "E3",
        "Model": (
            "ResNet-50 + SigLIP 2 + DINOv2 "
            "(Fixed Equal-Weight Fusion)"
        ),
        "Validation Accuracy (%)": round(E3_VAL_ACCURACY, 2),
        "Test Accuracy (%)": round(E3_TEST_ACCURACY, 2)
    }
])

e3_diagnostics.to_csv(
    E3_DIR / "experiment_3_diagnostics.csv",
    index=False
)

print("\nMODEL DIAGNOSTICS")
display(e3_diagnostics)

# ------------------------------------------------------------
# 6. Saving combined experiment results.
# ------------------------------------------------------------

E3_ARCHITECTURE = (
    "ResNet-50 (Places365) + "
    "SigLIP 2 Base (ViT-B/16, WebLI) + "
    "DINOv2 ViT-B/14 (LVD-142M)"
)

E3_METHOD = (
    "One-shot cosine similarity + "
    "fixed equal-weight three-model score fusion"
)

e3_results = pd.DataFrame([{
    "Experiment": "E3",
    "Architecture": E3_ARCHITECTURE,
    "Method": E3_METHOD,
    "Validation Accuracy (%)": round(E3_VAL_ACCURACY, 2),
    "Test Accuracy (%)": round(E3_TEST_ACCURACY, 2),
    "Runtime (minutes)": round(E3_RUNTIME / 60, 2)
}])

e3_results.to_csv(
    E3_DIR / "experiment_3_results.csv",
    index=False
)

# ------------------------------------------------------------
# 7. Saving test predictions and reference embeddings.
# ------------------------------------------------------------

torch.save({
    "test_labels": e3_rn_test_y,
    "test_predictions": e3_test_predictions,
    "validation_accuracy": E3_VAL_ACCURACY,
    "test_accuracy": E3_TEST_ACCURACY,
    "resnet_weight": E3_RN_WEIGHT,
    "siglip_weight": E3_SIG_WEIGHT,
    "dino_weight": E3_DINO_WEIGHT,
    "resnet_checkpoint": E3_RESNET_ID,
    "siglip_checkpoint": E3_SIGLIP_ID,
    "siglip_pretrained_weights": E3_SIGLIP_WEIGHTS,
    "dino_checkpoint": E3_DINO_ID,
    "cache_identity": E3_CACHE_ID,
    "class_names": CLASS_NAMES,
    "resnet_references": e3_rn_ref,
    "siglip_references": e3_sig_ref,
    "dino_references": e3_dino_ref,
    "reference_labels": e3_rn_ref_y
}, E3_DIR / "experiment_3_predictions.pt")

# ------------------------------------------------------------
# 8. Saving reproducibility configuration.
# ------------------------------------------------------------

e3_config = {
    "experiment": "E3",
    "architecture": E3_ARCHITECTURE,
    "method": E3_METHOD,
    "seed": SEED,
    "reference_images_per_class": 1,
    "total_reference_images": 16,
    "validation_images": 2384,
    "test_images": 400,
    "frozen_models": True,
    "training_epochs": 0,
    "resnet_checkpoint": E3_RESNET_ID,
    "siglip_checkpoint": E3_SIGLIP_ID,
    "siglip_pretrained": E3_SIGLIP_WEIGHTS,
    "dino_checkpoint": E3_DINO_ID,
    "dino_pretraining": "LVD-142M",
    "dino_feature": "Global CLS representation",
    "similarity": "cosine",
    "score_normalization": "per-image z-score",
    "fusion_weights": {
        "resnet": E3_RN_WEIGHT,
        "siglip": E3_SIG_WEIGHT,
        "dino": E3_DINO_WEIGHT
    },
    "validation_accuracy": E3_VAL_ACCURACY,
    "test_accuracy": E3_TEST_ACCURACY,
    "runtime_seconds": E3_RUNTIME,
    "runtime_note": (
        "Measured from initialization; "
        "cached features may reduce runtime."
    )
}

with open(
    E3_DIR / "experiment_3_config.json",
    "w"
) as f:
    json.dump(e3_config, f, indent=2)

# ------------------------------------------------------------
# 9. Updating common experiment tracking table.
# ------------------------------------------------------------

record_experiment(
    experiment="E3",
    architecture=E3_ARCHITECTURE,
    method=E3_METHOD,
    validation_accuracy=E3_VAL_ACCURACY,
    test_accuracy=E3_TEST_ACCURACY,
    runtime_seconds=E3_RUNTIME,
    observations=(
        "Three frozen pretrained models; "
        "16 one-shot references; "
        "fixed weights 1/3 each; "
        "no validation-based weight tuning."
    )
)

# ------------------------------------------------------------
# 10. Displaying final results.
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXPERIMENT 3 — FINAL COMBINED RESULTS")
print("=" * 65)

display(e3_results)

print(
    f"\nOverall Validation Accuracy: "
    f"{E3_VAL_ACCURACY:.2f}%"
)

print(
    f"Overall Test Accuracy: "
    f"{E3_TEST_ACCURACY:.2f}%"
)

print(
    f"Total Runtime: "
    f"{E3_RUNTIME / 60:.2f} minutes"
)

print("\nExperiment 3 completed successfully.")
print("All results have been saved.")

# ------------------------------------------------------------
# 11. Downloading result files.
# ------------------------------------------------------------

from google.colab import files

files.download(
    str(E3_DIR / "experiment_3_results.csv")
)

files.download(
    str(E3_DIR / "experiment_3_diagnostics.csv")
)

files.download(
    str(E3_DIR / "experiment_3_config.json")
)

files.download(
    str(ROOT / "experiment_results.csv")
)


EXTRACTING TEST FEATURES

Extracting features: resnet_test
resnet: 400/400 images
Saved feature cache: resnet_test

Extracting features: siglip_test
siglip: 400/400 images
Saved feature cache: siglip_test

Extracting features: dinov2_test
dino: 400/400 images
Saved feature cache: dinov2_test

INDIVIDUAL MODEL TEST ANALYSIS
ResNet-50: 83.50%
SigLIP 2: 78.75%
DINOv2: 79.00%

MODEL DIAGNOSTICS


,Experiment,Model,Validation Accuracy (%),Test Accuracy (%)
0,E3,ResNet-50 (Places365),81.67,83.50
1,E3,"SigLIP 2 Base (ViT-B/16, WebLI)",78.61,78.75
2,E3,DINOv2 ViT-B/14 (LVD-142M),75.29,79.00
3,E3,ResNet-50 + SigLIP 2 + DINOv2 (Fixed Equal-Wei...,82.63,83.50


/tmp/ipykernel_65882/1497323205.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat(


,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (seconds),Observations
0,E3,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + fixed equal-weigh...,82.6342,83.5,127.38,Three frozen pretrained models; 16 one-shot re...



EXPERIMENT 3 — FINAL COMBINED RESULTS


,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (minutes)
0,E3,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + fixed equal-weigh...,82.63,83.5,2.12


... (5 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Experiment 4: ResNet-50 (Places365) + SigLIP 2 + MoCo v2 with Multi-View Augmentation

**Objective:** Investigating whether deterministic multi-view feature aggregation improves one-shot image classification accuracy when combining supervised scene representations, vision-language representations, and self-supervised contrastive representations.

**Pretrained architectures:**
- ResNet-50 pretrained on Places365 for scene recognition.
- SigLIP 2 Base (ViT-B/16) pretrained on WebLI.
- MoCo v2 ResNet-50 pretrained on ImageNet-1K using momentum contrastive learning.

**Method:** Frozen pretrained feature extraction, deterministic multi-view augmentation, weighted feature aggregation, one-shot cosine similarity, per-image score standardization, and fixed equal-weight model fusion.

**One-shot restriction:** Exactly one labeled reference image per class is used. Augmented views are generated only from the same reference image. No additional labeled images are used for training, fine-tuning, or parameter optimization.

**Augmentation:** Five deterministic views are constructed for each image: original center view, horizontal flip, moderate center zoom, left spatial crop, and right spatial crop. Their predetermined aggregation weights are 0.40, 0.20, 0.20, 0.10, and 0.10, respectively.

**Model fusion:** Each model contributes exactly one-third of the final standardized similarity score. No fusion weights are optimized using validation or test labels.

**Evaluation:** Compare original-view and multi-view classification performance for individual models and the combined ensemble. The primary Experiment 4 result is the fixed-weight multi-view ensemble accuracy.

**Baseline:** Experiment 2 achieved 85.25% test accuracy using ResNet-50 and SigLIP 2. Experiment 3 achieved 83.50% test accuracy using ResNet-50, SigLIP 2, and DINOv2.

**Implementation:** All models remain frozen. Embeddings are cached separately for each model, image split, and augmentation setting to reduce repeated GPU computation.

### 4.1 Model Initialization and Deterministic Multi-View Augmentation

Initializing ResNet-50 with Places365 pretrained weights, SigLIP 2 Base with WebLI pretrained weights, and MoCo v2 ResNet-50 with its official self-supervised ImageNet pretrained checkpoint.

Removing the classification layers from the ResNet-based models to obtain global image embeddings. Using the visual encoder output for SigLIP 2.

Keeping all model parameters frozen and using evaluation mode throughout the experiment.

Constructing five deterministic image views:
1. Original center view — weight 0.40.
2. Horizontally flipped view — weight 0.20.
3. Moderate center zoom — weight 0.20.
4. Left spatial crop — weight 0.10.
5. Right spatial crop — weight 0.10.

Applying each model's required normalization and preprocessing. Extracting and aggregating features without modifying pretrained model parameters.

Verifying the dataset splits and creating experiment-specific cache identifiers to prevent mixing results with previous experiments.

In [10]:
%pip install -q open_clip_torch

In [12]:
# ============================================================
# EXPERIMENT 4 — INITIALIZATION AND MULTI-VIEW AUGMENTATION
# ResNet-50 Places365 + SigLIP 2 + MoCo v2 ResNet-50
# ============================================================

import gc
import json
import time
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from PIL import Image
from torchvision import transforms
from torchvision.transforms import functional as TF
from IPython.display import display

import open_clip

E4_START = time.time()
E4_DIR = ROOT / "experiment_4"
E4_DIR.mkdir(parents=True, exist_ok=True)

E4_RESNET_ID = "resnet50_places365"
E4_SIGLIP_ID = "ViT-B-16-SigLIP2"
E4_SIGLIP_WEIGHTS = "webli"
E4_MOCO_ID = "moco_v2_resnet50_200ep"

E4_RESNET_URL = (
    "http://places2.csail.mit.edu/models_places365/"
    "resnet50_places365.pth.tar"
)

E4_MOCO_URL = (
    "https://dl.fbaipublicfiles.com/moco/"
    "moco_checkpoints/moco_v2_200ep/"
    "moco_v2_200ep_pretrain.pth.tar"
)

E4_RESNET_FILE = E4_DIR / "resnet50_places365.pth.tar"
E4_MOCO_FILE = E4_DIR / "moco_v2_200ep.pth.tar"

E4_VIEW_NAMES = [
    "original",
    "horizontal_flip",
    "center_zoom",
    "left_crop",
    "right_crop"
]

E4_VIEW_WEIGHTS = torch.tensor(
    [0.40, 0.20, 0.20, 0.10, 0.10],
    dtype=torch.float32
)

assert len(E4_VIEW_NAMES) == 5
assert torch.isclose(
    E4_VIEW_WEIGHTS.sum(),
    torch.tensor(1.0)
)

assert torch.cuda.is_available(), (
    "Enable a GPU in Colab before running Experiment 4."
)

assert len(CLASS_NAMES) == 16
assert len(ONE_SHOT_SAMPLES) == 16
assert len(VALIDATION_SAMPLES) == 2384
assert len(TEST_SAMPLES) == 400

assert sorted(
    int(label) for _, label in ONE_SHOT_SAMPLES
) == list(range(16))

def e4_paths(samples):
    return {
        str(Path(path).resolve())
        for path, _ in samples
    }

assert e4_paths(ONE_SHOT_SAMPLES).isdisjoint(
    e4_paths(VALIDATION_SAMPLES)
)

assert e4_paths(TEST_SAMPLES).isdisjoint(
    e4_paths(ONE_SHOT_SAMPLES)
    | e4_paths(VALIDATION_SAMPLES)
)

# ------------------------------------------------------------
# 1. Checkpoint downloader with partial-file protection.
# ------------------------------------------------------------

def e4_download_checkpoint(url, destination):
    if destination.exists():
        if destination.stat().st_size > 10_000_000:
            print("Using checkpoint:", destination.name)
            return
        destination.unlink()

    temporary = destination.with_suffix(
        destination.suffix + ".partial"
    )

    temporary.unlink(missing_ok=True)

    print("Downloading:", destination.name)

    try:
        urllib.request.urlretrieve(
            url,
            str(temporary)
        )

        if temporary.stat().st_size < 10_000_000:
            raise RuntimeError(
                "Downloaded checkpoint is unexpectedly small."
            )

        temporary.replace(destination)

    except Exception:
        temporary.unlink(missing_ok=True)
        raise


# ------------------------------------------------------------
# 2. ResNet-50 Places365.
# ------------------------------------------------------------

print("\nInitializing ResNet-50 Places365...")

e4_download_checkpoint(
    E4_RESNET_URL,
    E4_RESNET_FILE
)

e4_resnet = models.resnet50(
    weights=None,
    num_classes=365
)

e4_checkpoint = torch.load(
    E4_RESNET_FILE,
    map_location="cpu",
    weights_only=False
)

e4_state = e4_checkpoint.get(
    "state_dict",
    e4_checkpoint
)

e4_state = {
    key.removeprefix("module."): value
    for key, value in e4_state.items()
}

e4_resnet.load_state_dict(
    e4_state,
    strict=True
)

e4_resnet.fc = nn.Identity()
e4_resnet.eval()
e4_resnet.requires_grad_(False)

del e4_checkpoint, e4_state

print("ResNet-50 Places365 ready.")

# ------------------------------------------------------------
# 3. SigLIP 2 Base.
# ------------------------------------------------------------

print("\nInitializing SigLIP 2 Base...")

e4_siglip, _, e4_siglip_transform = (
    open_clip.create_model_and_transforms(
        E4_SIGLIP_ID,
        pretrained=E4_SIGLIP_WEIGHTS,
        device="cpu"
    )
)

e4_siglip.eval()
e4_siglip.requires_grad_(False)

print("SigLIP 2 Base ready.")

# ------------------------------------------------------------
# 4. Official MoCo v2 ResNet-50 checkpoint.
# ------------------------------------------------------------

print("\nInitializing MoCo v2 ResNet-50...")

e4_download_checkpoint(
    E4_MOCO_URL,
    E4_MOCO_FILE
)

e4_moco = models.resnet50(
    weights=None,
    num_classes=1000
)

e4_checkpoint = torch.load(
    E4_MOCO_FILE,
    map_location="cpu",
    weights_only=False
)

e4_state = e4_checkpoint["state_dict"]

# Official MoCo checkpoint includes query and key encoders.
# Use only the pretrained query encoder backbone.
# Exclude its projection head (fc).

e4_moco_state = {}

for key, value in e4_state.items():
    prefix = "module.encoder_q."

    if key.startswith(prefix):
        new_key = key[len(prefix):]

        if not new_key.startswith("fc."):
            e4_moco_state[new_key] = value

assert len(e4_moco_state) > 100, (
    "MoCo checkpoint did not contain the expected "
    "query encoder backbone."
)

e4_moco.fc = nn.Identity()

e4_moco.load_state_dict(
    e4_moco_state,
    strict=True
)

e4_moco.eval()
e4_moco.requires_grad_(False)

del e4_checkpoint, e4_state, e4_moco_state

print("MoCo v2 ResNet-50 ready.")

# ------------------------------------------------------------
# 5. Model-specific normalization.
# ------------------------------------------------------------

E4_IMAGENET_MEAN = [0.485, 0.456, 0.406]
E4_IMAGENET_STD = [0.229, 0.224, 0.225]

# Use the same baseline preprocessing as Experiment 3
# for ResNet-50 Places365.

e4_resnet_original = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        E4_IMAGENET_MEAN,
        E4_IMAGENET_STD
    )
])

# Standard ImageNet-style evaluation preprocessing
# for the MoCo v2 backbone.

e4_moco_original = transforms.Compose([
    transforms.Resize(
        256,
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        E4_IMAGENET_MEAN,
        E4_IMAGENET_STD
    )
])

# ------------------------------------------------------------
# 6. Five-view augmentation.
# Each view is created from the original RGB image.
# ------------------------------------------------------------

def e4_generate_views(image):
    image = image.convert("RGB")

    width, height = image.size

    # Original image.
    original = image

    # Horizontal reflection.
    flipped = TF.hflip(image)

    # Moderate zoom: central 85% of the image.
    zoom_width = max(1, round(width * 0.85))
    zoom_height = max(1, round(height * 0.85))

    zoom_left = (width - zoom_width) // 2
    zoom_top = (height - zoom_height) // 2

    center_zoom = TF.crop(
        image,
        zoom_top,
        zoom_left,
        zoom_height,
        zoom_width
    )

    # Left and right crops preserve 85% image width.
    crop_width = max(1, round(width * 0.85))

    left_crop = TF.crop(
        image,
        0,
        0,
        height,
        crop_width
    )

    right_crop = TF.crop(
        image,
        0,
        width - crop_width,
        height,
        crop_width
    )

    return [
        original,
        flipped,
        center_zoom,
        left_crop,
        right_crop
    ]


class E4MultiViewTransform:
    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, image):
        views = e4_generate_views(image)

        return torch.stack([
            self.base_transform(view)
            for view in views
        ], dim=0)


e4_resnet_multiview = E4MultiViewTransform(
    e4_resnet_original
)

e4_siglip_multiview = E4MultiViewTransform(
    e4_siglip_transform
)

e4_moco_multiview = E4MultiViewTransform(
    e4_moco_original
)

# ------------------------------------------------------------
# 7. Cache identity.
# ------------------------------------------------------------

e4_split_signature = [
    sorted(
        (str(Path(path).resolve()), int(label))
        for path, label in samples
    )
    for samples in (
        ONE_SHOT_SAMPLES,
        VALIDATION_SAMPLES,
        TEST_SAMPLES
    )
]

E4_CACHE_ID = hashlib.sha256(
    json.dumps({
        "splits": e4_split_signature,
        "models": [
            E4_RESNET_ID,
            E4_SIGLIP_ID,
            E4_SIGLIP_WEIGHTS,
            E4_MOCO_ID
        ],
        "view_names": E4_VIEW_NAMES,
        "view_weights": E4_VIEW_WEIGHTS.tolist(),
        "version": "e4_multiview_v1"
    }, sort_keys=True).encode()
).hexdigest()[:16]

# ------------------------------------------------------------
# 8. Verifying all pretrained backbones are frozen.
# ------------------------------------------------------------

for model in (
    e4_resnet,
    e4_siglip,
    e4_moco
):
    assert not any(
        parameter.requires_grad
        for parameter in model.parameters()
    )

print("\n" + "=" * 65)
print("EXPERIMENT 4 — INITIALIZATION SUCCESSFUL")
print("=" * 65)

print("ResNet:", E4_RESNET_ID)
print("SigLIP:", E4_SIGLIP_ID)
print("MoCo:", E4_MOCO_ID)
print("Device:", DEVICE)
print("Reference images:", len(ONE_SHOT_SAMPLES))
print("Validation images:", len(VALIDATION_SAMPLES))
print("Test images:", len(TEST_SAMPLES))
print("Augmented views:", len(E4_VIEW_NAMES))
print("View weights:", E4_VIEW_WEIGHTS.tolist())
print("Fusion weights: 1/3 + 1/3 + 1/3")
print("All models frozen: Yes")
print("Cache identity:", E4_CACHE_ID)

print("\nReady for Experiment 4 Code Cell 2.")


Initializing ResNet-50 Places365...
Using checkpoint: resnet50_places365.pth.tar
ResNet-50 Places365 ready.

Initializing SigLIP 2 Base...


SigLIP 2 Base ready.

Initializing MoCo v2 ResNet-50...
Downloading: moco_v2_200ep.pth.tar
MoCo v2 ResNet-50 ready.

EXPERIMENT 4 — INITIALIZATION SUCCESSFUL
ResNet: resnet50_places365
SigLIP: ViT-B-16-SigLIP2
MoCo: moco_v2_resnet50_200ep
Device: cuda
Reference images: 16
Validation images: 2384
Test images: 400
Augmented views: 5
View weights: [0.4000000059604645, 0.20000000298023224, 0.20000000298023224, 0.10000000149011612, 0.10000000149011612]
Fusion weights: 1/3 + 1/3 + 1/3
All models frozen: Yes
Cache identity: 82be8ffeb62dfce0

Ready for Experiment 4 Code Cell 2.


### 4.2 Multi-View Feature Extraction and Validation Evaluation

Extracting original-view and five-view embeddings for the 16 one-shot reference images and the validation images using all three pretrained models.

For multi-view evaluation, normalizing the embedding of each augmented view, combining the five embeddings using predetermined weights of 0.40, 0.20, 0.20, 0.10, and 0.10, and normalizing the aggregated embedding again.

Using cosine similarity between each evaluation embedding and the 16 reference embeddings to obtain class similarity scores.

Standardizing class scores independently for each model and image. Combining the three models using fixed equal weights of one-third each.

Calculating original-view and multi-view validation accuracies for each model and for the combined ensemble. These results are reported for diagnostic analysis only.

In [13]:
# ============================================================
# EXPERIMENT 4 — FEATURE EXTRACTION AND VALIDATION
# Original view versus center-weighted five-view aggregation
# ============================================================

def e4_extract_features(
    model,
    model_type,
    samples,
    transform,
    multiview=False,
    batch_size=16
):
    loader = create_loader(
        samples,
        transform,
        batch_size=batch_size
    )

    features = []
    labels = []

    model = model.to(DEVICE).eval()

    try:
        with torch.inference_mode():

            for step, (images, targets) in enumerate(loader):

                if multiview:
                    # Shape: [batch, 5, 3, 224, 224]
                    batch_size_actual, num_views = (
                        images.shape[:2]
                    )

                    assert num_views == 5

                    # Process one view at a time to avoid
                    # multiplying peak GPU memory usage.
                    view_features = []

                    for view_index in range(num_views):
                        view_images = images[
                            :, view_index
                        ].to(
                            DEVICE,
                            non_blocking=True
                        )

                        with torch.autocast(
                            device_type="cuda",
                            dtype=torch.float16
                        ):
                            if model_type == "siglip":
                                embeddings = (
                                    model.encode_image(
                                        view_images
                                    )
                                )
                            else:
                                embeddings = model(
                                    view_images
                                )

                        embeddings = F.normalize(
                            embeddings.float(),
                            p=2,
                            dim=1
                        )

                        view_features.append(embeddings)

                    stacked = torch.stack(
                        view_features,
                        dim=1
                    )

                    weights = E4_VIEW_WEIGHTS.to(
                        device=stacked.device,
                        dtype=stacked.dtype
                    )

                    aggregated = (
                        stacked
                        * weights.view(1, 5, 1)
                    ).sum(dim=1)

                    embeddings = F.normalize(
                        aggregated,
                        p=2,
                        dim=1
                    )

                    del stacked, view_features

                else:
                    images = images.to(
                        DEVICE,
                        non_blocking=True
                    )

                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16
                    ):
                        if model_type == "siglip":
                            embeddings = (
                                model.encode_image(images)
                            )
                        else:
                            embeddings = model(images)

                    embeddings = F.normalize(
                        embeddings.float(),
                        p=2,
                        dim=1
                    )

                features.append(
                    embeddings.detach().cpu()
                )

                labels.append(
                    targets.detach().cpu()
                )

                if (step + 1) % 25 == 0:
                    processed = min(
                        (step + 1) * batch_size,
                        len(samples)
                    )

                    print(
                        f"{model_type} "
                        f"{'multiview' if multiview else 'original'}: "
                        f"{processed}/{len(samples)}"
                    )

    finally:
        model.cpu()
        cleanup_gpu()

    return (
        torch.cat(features, dim=0),
        torch.cat(labels, dim=0)
    )


def e4_get_features(
    name,
    model,
    model_type,
    samples,
    transform,
    multiview=False
):
    cache_path = (
        E4_DIR / f"{name}_{E4_CACHE_ID}.pt"
    )

    expected_labels = torch.tensor(
        [int(label) for _, label in samples],
        dtype=torch.long
    )

    if cache_path.exists():
        try:
            cached = torch.load(
                cache_path,
                map_location="cpu",
                weights_only=True
            )

            cached_features = cached["features"]
            cached_labels = cached["labels"]

            if (
                cached_features.shape[0] == len(samples)
                and torch.equal(
                    cached_labels,
                    expected_labels
                )
                and torch.isfinite(
                    cached_features
                ).all()
            ):
                print("Loading cached features:", name)

                return (
                    cached_features,
                    cached_labels
                )

            print("Invalid cache, recomputing:", name)

        except Exception:
            print("Unreadable cache, recomputing:", name)

    print("\nExtracting:", name)

    features, labels = e4_extract_features(
        model=model,
        model_type=model_type,
        samples=samples,
        transform=transform,
        multiview=multiview,
        batch_size=16
    )

    assert torch.equal(
        labels,
        expected_labels
    )

    assert torch.isfinite(features).all()

    torch.save(
        {
            "features": features,
            "labels": labels
        },
        cache_path
    )

    print("Saved feature cache:", name)

    return features, labels


def e4_class_scores(
    features,
    references,
    reference_labels
):
    order = torch.argsort(reference_labels)

    assert torch.equal(
        reference_labels[order],
        torch.arange(len(CLASS_NAMES))
    )

    reference_matrix = references[order]

    return features @ reference_matrix.T


def e4_normalize_scores(scores):
    mean = scores.mean(
        dim=1,
        keepdim=True
    )

    std = scores.std(
        dim=1,
        keepdim=True,
        unbiased=False
    ).clamp_min(1e-6)

    return (scores - mean) / std


def e4_accuracy(scores, labels):
    predictions = scores.argmax(dim=1)

    return (
        (predictions == labels)
        .float()
        .mean()
        .item() * 100
    )


def e4_fuse_scores(score_dict):
    assert set(score_dict) == {
        "resnet", "siglip", "moco"
    }

    return (
        e4_normalize_scores(score_dict["resnet"])
        +
        e4_normalize_scores(score_dict["siglip"])
        +
        e4_normalize_scores(score_dict["moco"])
    ) / 3.0


# ------------------------------------------------------------
# 1. Defining models and transformations.
# ------------------------------------------------------------

E4_MODEL_CONFIGS = {
    "resnet": {
        "model": e4_resnet,
        "type": "resnet",
        "original": e4_resnet_original,
        "multiview": e4_resnet_multiview
    },
    "siglip": {
        "model": e4_siglip,
        "type": "siglip",
        "original": e4_siglip_transform,
        "multiview": e4_siglip_multiview
    },
    "moco": {
        "model": e4_moco,
        "type": "moco",
        "original": e4_moco_original,
        "multiview": e4_moco_multiview
    }
}

E4_MODES = ["original", "multiview"]

# ------------------------------------------------------------
# 2. Extracting reference and validation features.
# Each model is processed independently.
# ------------------------------------------------------------

e4_reference_features = {}
e4_validation_features = {}
e4_reference_labels = {}
e4_validation_labels = {}

for model_name, config in E4_MODEL_CONFIGS.items():

    print("\n" + "=" * 60)
    print("PROCESSING MODEL:", model_name.upper())
    print("=" * 60)

    for mode in E4_MODES:

        is_multiview = mode == "multiview"

        ref_features, ref_labels = e4_get_features(
            name=f"{model_name}_{mode}_reference",
            model=config["model"],
            model_type=config["type"],
            samples=ONE_SHOT_SAMPLES,
            transform=config[mode],
            multiview=is_multiview
        )

        val_features, val_labels = e4_get_features(
            name=f"{model_name}_{mode}_validation",
            model=config["model"],
            model_type=config["type"],
            samples=VALIDATION_SAMPLES,
            transform=config[mode],
            multiview=is_multiview
        )

        key = (model_name, mode)

        e4_reference_features[key] = ref_features
        e4_validation_features[key] = val_features

        e4_reference_labels[key] = ref_labels
        e4_validation_labels[key] = val_labels


# ------------------------------------------------------------
# 3. Computing individual model validation scores.
# ------------------------------------------------------------

e4_validation_scores = {
    "original": {},
    "multiview": {}
}

e4_validation_accuracies = {
    "original": {},
    "multiview": {}
}

e4_ground_truth = torch.tensor(
    [int(label) for _, label in VALIDATION_SAMPLES],
    dtype=torch.long
)

for mode in E4_MODES:

    for model_name in E4_MODEL_CONFIGS:

        key = (model_name, mode)

        assert torch.equal(
            e4_validation_labels[key],
            e4_ground_truth
        )

        scores = e4_class_scores(
            e4_validation_features[key],
            e4_reference_features[key],
            e4_reference_labels[key]
        )

        e4_validation_scores[mode][model_name] = scores

        e4_validation_accuracies[mode][model_name] = (
            e4_accuracy(
                scores,
                e4_ground_truth
            )
        )


# ------------------------------------------------------------
# 4. Fixed equal-weight ensemble fusion.
# ------------------------------------------------------------

e4_fused_validation_scores = {}
e4_fused_validation_accuracies = {}

for mode in E4_MODES:

    fused_scores = e4_fuse_scores(
        e4_validation_scores[mode]
    )

    e4_fused_validation_scores[mode] = fused_scores

    e4_fused_validation_accuracies[mode] = (
        e4_accuracy(
            fused_scores,
            e4_ground_truth
        )
    )


# ------------------------------------------------------------
# 5. Displaying original versus multi-view validation.
# ------------------------------------------------------------

e4_validation_rows = []

for model_name in ["resnet", "siglip", "moco"]:

    original_accuracy = (
        e4_validation_accuracies["original"][model_name]
    )

    multiview_accuracy = (
        e4_validation_accuracies["multiview"][model_name]
    )

    e4_validation_rows.append({
        "Model": model_name.upper(),
        "Original Validation (%)": round(
            original_accuracy, 2
        ),
        "Multi-View Validation (%)": round(
            multiview_accuracy, 2
        ),
        "Change (pp)": round(
            multiview_accuracy - original_accuracy,
            2
        )
    })

original_fused = (
    e4_fused_validation_accuracies["original"]
)

multiview_fused = (
    e4_fused_validation_accuracies["multiview"]
)

e4_validation_rows.append({
    "Model": "THREE-MODEL ENSEMBLE",
    "Original Validation (%)": round(
        original_fused, 2
    ),
    "Multi-View Validation (%)": round(
        multiview_fused, 2
    ),
    "Change (pp)": round(
        multiview_fused - original_fused,
        2
    )
})

e4_validation_df = pd.DataFrame(
    e4_validation_rows
)

print("\n" + "=" * 65)
print("EXPERIMENT 4 — VALIDATION RESULTS")
print("=" * 65)

display(e4_validation_df)

E4_VAL_ACCURACY = multiview_fused

print(
    f"\nPrimary Multi-View Validation Accuracy: "
    f"{E4_VAL_ACCURACY:.2f}%"
)

print("Fusion weights: 1/3 + 1/3 + 1/3")
print("No validation-based parameter optimization.")

e4_validation_df.to_csv(
    E4_DIR / "experiment_4_validation_diagnostics.csv",
    index=False
)

print("\nReady for Experiment 4 Code Cell 3.")


PROCESSING MODEL: RESNET

Extracting: resnet_original_reference
Saved feature cache: resnet_original_reference

Extracting: resnet_original_validation
resnet original: 400/2384
resnet original: 800/2384
resnet original: 1200/2384
resnet original: 1600/2384
resnet original: 2000/2384
Saved feature cache: resnet_original_validation

Extracting: resnet_multiview_reference
Saved feature cache: resnet_multiview_reference

Extracting: resnet_multiview_validation
resnet multiview: 400/2384
resnet multiview: 800/2384
resnet multiview: 1200/2384
resnet multiview: 1600/2384
resnet multiview: 2000/2384
Saved feature cache: resnet_multiview_validation

PROCESSING MODEL: SIGLIP

Extracting: siglip_original_reference
Saved feature cache: siglip_original_reference

Extracting: siglip_original_validation
siglip original: 400/2384
siglip original: 800/2384
siglip original: 1200/2384
siglip original: 1600/2384
siglip original: 2000/2384
Saved feature cache: siglip_original_validation

Extracting: sigli

,Model,Original Validation (%),Multi-View Validation (%),Change (pp)
0,RESNET,81.67,82.13,0.46
1,SIGLIP,78.61,77.52,-1.09
2,MOCO,71.14,70.89,-0.25
3,THREE-MODEL ENSEMBLE,83.10,82.30,-0.80



Primary Multi-View Validation Accuracy: 82.30%
Fusion weights: 1/3 + 1/3 + 1/3
No validation-based parameter optimization.

Ready for Experiment 4 Code Cell 3.


### 4.3 Test Evaluation and Final Results

Evaluating the three frozen pretrained models on the separate 400-image test dataset using the same one-shot reference images and predetermined augmentation settings.

Calculating original-view and multi-view cosine similarity scores for each model. Apply per-image score standardization and combine the model scores using fixed equal weights.

Reporting individual model test accuracies and comparing the original-view ensemble against the multi-view ensemble to measure the effect of augmentation.

The primary Experiment 4 result is the multi-view ensemble test accuracy. The original-view ensemble serves as an ablation baseline.

Saving the validation and test diagnostics, experiment results, predictions, configuration, and feature caches for reproducibility.

Comparing Experiment 4 against the previously completed experiments without modifying augmentation parameters or fusion weights based on the observed evaluation results.

In [14]:
# ============================================================
# EXPERIMENT 4 — TEST EVALUATION AND FINAL RESULTS
# Original versus multi-view, fixed equal-weight fusion
# ============================================================

# ------------------------------------------------------------
# 1. Extracting original and multi-view test features.
# ------------------------------------------------------------

e4_test_features = {}
e4_test_labels = {}

for model_name, config in E4_MODEL_CONFIGS.items():

    print("\n" + "=" * 60)
    print("TEST MODEL:", model_name.upper())
    print("=" * 60)

    for mode in E4_MODES:

        features, labels = e4_get_features(
            name=f"{model_name}_{mode}_test",
            model=config["model"],
            model_type=config["type"],
            samples=TEST_SAMPLES,
            transform=config[mode],
            multiview=(mode == "multiview")
        )

        key = (model_name, mode)

        e4_test_features[key] = features
        e4_test_labels[key] = labels


# ------------------------------------------------------------
# 2. Computing individual model test scores.
# ------------------------------------------------------------

e4_test_ground_truth = torch.tensor(
    [int(label) for _, label in TEST_SAMPLES],
    dtype=torch.long
)

e4_test_scores = {
    "original": {},
    "multiview": {}
}

e4_test_accuracies = {
    "original": {},
    "multiview": {}
}

for mode in E4_MODES:

    for model_name in E4_MODEL_CONFIGS:

        key = (model_name, mode)

        assert torch.equal(
            e4_test_labels[key],
            e4_test_ground_truth
        )

        scores = e4_class_scores(
            e4_test_features[key],
            e4_reference_features[key],
            e4_reference_labels[key]
        )

        e4_test_scores[mode][model_name] = scores

        e4_test_accuracies[mode][model_name] = (
            e4_accuracy(
                scores,
                e4_test_ground_truth
            )
        )


# ------------------------------------------------------------
# 3. Applying fixed three-model fusion.
# ------------------------------------------------------------

e4_fused_test_scores = {}
e4_fused_test_accuracies = {}
e4_test_predictions = {}

for mode in E4_MODES:

    fused = e4_fuse_scores(
        e4_test_scores[mode]
    )

    e4_fused_test_scores[mode] = fused

    e4_test_predictions[mode] = fused.argmax(
        dim=1
    )

    e4_fused_test_accuracies[mode] = (
        e4_accuracy(
            fused,
            e4_test_ground_truth
        )
    )


# ------------------------------------------------------------
# 4. Building individual model diagnostics.
# ------------------------------------------------------------

e4_diagnostic_rows = []

for model_name in ["resnet", "siglip", "moco"]:

    for mode in E4_MODES:

        e4_diagnostic_rows.append({
            "Experiment": "E4",
            "Model": model_name.upper(),
            "Evaluation Mode": mode,
            "Validation Accuracy (%)": round(
                e4_validation_accuracies[mode][model_name],
                2
            ),
            "Test Accuracy (%)": round(
                e4_test_accuracies[mode][model_name],
                2
            )
        })

for mode in E4_MODES:

    e4_diagnostic_rows.append({
        "Experiment": "E4",
        "Model": "THREE-MODEL ENSEMBLE",
        "Evaluation Mode": mode,
        "Validation Accuracy (%)": round(
            e4_fused_validation_accuracies[mode],
            2
        ),
        "Test Accuracy (%)": round(
            e4_fused_test_accuracies[mode],
            2
        )
    })

e4_diagnostics = pd.DataFrame(
    e4_diagnostic_rows
)

e4_diagnostics.to_csv(
    E4_DIR / "experiment_4_diagnostics.csv",
    index=False
)

print("\nINDIVIDUAL MODEL AND ENSEMBLE DIAGNOSTICS")
display(e4_diagnostics)


# ------------------------------------------------------------
# 5. Calculating augmentation effect.
# ------------------------------------------------------------

E4_ORIGINAL_VAL_ACCURACY = (
    e4_fused_validation_accuracies["original"]
)

E4_ORIGINAL_TEST_ACCURACY = (
    e4_fused_test_accuracies["original"]
)

E4_VAL_ACCURACY = (
    e4_fused_validation_accuracies["multiview"]
)

E4_TEST_ACCURACY = (
    e4_fused_test_accuracies["multiview"]
)

E4_RUNTIME = time.time() - E4_START

e4_augmentation_effect = pd.DataFrame([
    {
        "Evaluation": "Validation",
        "Original (%)": round(
            E4_ORIGINAL_VAL_ACCURACY, 2
        ),
        "Multi-View (%)": round(
            E4_VAL_ACCURACY, 2
        ),
        "Change (pp)": round(
            E4_VAL_ACCURACY - E4_ORIGINAL_VAL_ACCURACY,
            2
        )
    },
    {
        "Evaluation": "Test",
        "Original (%)": round(
            E4_ORIGINAL_TEST_ACCURACY, 2
        ),
        "Multi-View (%)": round(
            E4_TEST_ACCURACY, 2
        ),
        "Change (pp)": round(
            E4_TEST_ACCURACY - E4_ORIGINAL_TEST_ACCURACY,
            2
        )
    }
])

e4_augmentation_effect.to_csv(
    E4_DIR / "experiment_4_augmentation_effect.csv",
    index=False
)

print("\nAUGMENTATION EFFECT")
display(e4_augmentation_effect)


# ------------------------------------------------------------
# 6. Saving primary experiment results.
# ------------------------------------------------------------

E4_ARCHITECTURE = (
    "ResNet-50 (Places365) + "
    "SigLIP 2 Base (ViT-B/16, WebLI) + "
    "MoCo v2 ResNet-50 (ImageNet-1K)"
)

E4_METHOD = (
    "One-shot cosine similarity + "
    "five-view center-weighted feature aggregation + "
    "fixed equal-weight three-model fusion"
)

e4_results = pd.DataFrame([{
    "Experiment": "E4",
    "Architecture": E4_ARCHITECTURE,
    "Method": E4_METHOD,
    "Validation Accuracy (%)": round(
        E4_VAL_ACCURACY, 2
    ),
    "Test Accuracy (%)": round(
        E4_TEST_ACCURACY, 2
    ),
    "Runtime (minutes)": round(
        E4_RUNTIME / 60, 2
    )
}])

e4_results.to_csv(
    E4_DIR / "experiment_4_results.csv",
    index=False
)


# ------------------------------------------------------------
# 7. Saving predictions and one-shot reference embeddings.
# ------------------------------------------------------------

torch.save({
    "test_labels": e4_test_ground_truth,
    "original_test_predictions": (
        e4_test_predictions["original"]
    ),
    "multiview_test_predictions": (
        e4_test_predictions["multiview"]
    ),
    "original_test_scores": (
        e4_fused_test_scores["original"]
    ),
    "multiview_test_scores": (
        e4_fused_test_scores["multiview"]
    ),
    "reference_features": e4_reference_features,
    "reference_labels": e4_reference_labels,
    "class_names": CLASS_NAMES,
    "view_names": E4_VIEW_NAMES,
    "view_weights": E4_VIEW_WEIGHTS,
    "cache_identity": E4_CACHE_ID
}, E4_DIR / "experiment_4_predictions.pt")


# ------------------------------------------------------------
# 8. Saving reproducibility configuration.
# ------------------------------------------------------------

e4_config = {
    "experiment": "E4",
    "architecture": E4_ARCHITECTURE,
    "method": E4_METHOD,
    "seed": SEED,
    "reference_images_per_class": 1,
    "total_reference_images": 16,
    "validation_images": 2384,
    "test_images": 400,
    "models": {
        "resnet": E4_RESNET_ID,
        "siglip": E4_SIGLIP_ID,
        "siglip_weights": E4_SIGLIP_WEIGHTS,
        "moco": E4_MOCO_ID
    },
    "frozen_models": True,
    "training_epochs": 0,
    "augmentation": {
        "type": "deterministic_multiview",
        "view_names": E4_VIEW_NAMES,
        "view_weights": E4_VIEW_WEIGHTS.tolist(),
        "aggregation": (
            "L2-normalize individual embeddings, "
            "weighted average, L2-normalize again"
        )
    },
    "similarity": "cosine",
    "score_normalization": "per-image z-score",
    "model_fusion_weights": {
        "resnet": 1.0 / 3.0,
        "siglip": 1.0 / 3.0,
        "moco": 1.0 / 3.0
    },
    "original_validation_accuracy": (
        E4_ORIGINAL_VAL_ACCURACY
    ),
    "original_test_accuracy": (
        E4_ORIGINAL_TEST_ACCURACY
    ),
    "multiview_validation_accuracy": (
        E4_VAL_ACCURACY
    ),
    "multiview_test_accuracy": (
        E4_TEST_ACCURACY
    ),
    "runtime_seconds": E4_RUNTIME,
    "cache_identity": E4_CACHE_ID,
    "runtime_note": (
        "Runtime measured from initialization. "
        "Cached embeddings may reduce runtime."
    )
}

with open(
    E4_DIR / "experiment_4_config.json",
    "w"
) as f:
    json.dump(
        e4_config,
        f,
        indent=2
    )


# ------------------------------------------------------------
# 9. Updating common experiment tracking table.
# ------------------------------------------------------------

record_experiment(
    experiment="E4",
    architecture=E4_ARCHITECTURE,
    method=E4_METHOD,
    validation_accuracy=E4_VAL_ACCURACY,
    test_accuracy=E4_TEST_ACCURACY,
    runtime_seconds=E4_RUNTIME,
    observations=(
        "Three frozen pretrained models; "
        "five deterministic views; "
        "center-weighted feature aggregation; "
        "fixed equal-weight model fusion; "
        "no fine-tuning or validation-based tuning."
    )
)


# ------------------------------------------------------------
# 10. Displaying final results.
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXPERIMENT 4 — FINAL RESULTS")
print("=" * 65)

display(e4_results)

print(
    f"\nOriginal-View Validation Accuracy: "
    f"{E4_ORIGINAL_VAL_ACCURACY:.2f}%"
)

print(
    f"Original-View Test Accuracy: "
    f"{E4_ORIGINAL_TEST_ACCURACY:.2f}%"
)

print(
    f"\nMulti-View Validation Accuracy: "
    f"{E4_VAL_ACCURACY:.2f}%"
)

print(
    f"Multi-View Test Accuracy: "
    f"{E4_TEST_ACCURACY:.2f}%"
)

print(
    f"\nTest Accuracy Change: "
    f"{E4_TEST_ACCURACY - E4_ORIGINAL_TEST_ACCURACY:+.2f} "
    "percentage points"
)

print(
    f"Total Runtime: "
    f"{E4_RUNTIME / 60:.2f} minutes"
)

print("\nExperiment 4 completed successfully.")
print("All results saved in:", E4_DIR)


# ------------------------------------------------------------
# 11. Downloading result files.
# ------------------------------------------------------------

from google.colab import files

for filename in [
    "experiment_4_results.csv",
    "experiment_4_diagnostics.csv",
    "experiment_4_augmentation_effect.csv",
    "experiment_4_config.json"
]:
    files.download(
        str(E4_DIR / filename)
    )

common_results_path = (
    ROOT / "experiment_results.csv"
)

if common_results_path.exists():
    files.download(
        str(common_results_path)
    )


TEST MODEL: RESNET

Extracting: resnet_original_test
resnet original: 400/400
Saved feature cache: resnet_original_test

Extracting: resnet_multiview_test
resnet multiview: 400/400
Saved feature cache: resnet_multiview_test

TEST MODEL: SIGLIP

Extracting: siglip_original_test
siglip original: 400/400
Saved feature cache: siglip_original_test

Extracting: siglip_multiview_test
siglip multiview: 400/400
Saved feature cache: siglip_multiview_test

TEST MODEL: MOCO

Extracting: moco_original_test
moco original: 400/400
Saved feature cache: moco_original_test

Extracting: moco_multiview_test
moco multiview: 400/400
Saved feature cache: moco_multiview_test

INDIVIDUAL MODEL AND ENSEMBLE DIAGNOSTICS


,Experiment,Model,Evaluation Mode,Validation Accuracy (%),Test Accuracy (%)
0,E4,RESNET,original,81.67,83.50
1,E4,RESNET,multiview,82.13,83.25
2,E4,SIGLIP,original,78.61,78.75
3,E4,SIGLIP,multiview,77.52,78.50
4,E4,MOCO,original,71.14,73.75
5,E4,MOCO,multiview,70.89,72.00
6,E4,THREE-MODEL ENSEMBLE,original,83.10,84.50
7,E4,THREE-MODEL ENSEMBLE,multiview,82.30,82.75



AUGMENTATION EFFECT


,Evaluation,Original (%),Multi-View (%),Change (pp)
0,Validation,83.1,82.30,-0.80
1,Test,84.5,82.75,-1.75


/tmp/ipykernel_372/1497323205.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat(


,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (seconds),Observations
0,E4,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + five-view center-...,82.2987,82.75,288.21,Three frozen pretrained models; five determini...


... (9 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Experiment 5: ResNet-50 (Places365) + SigLIP 2 with Reference-Only Prototype Refinement

**Objective:** Investigating whether mild reference-only augmentation improves one-shot image classification by constructing more robust class prototypes from the single labeled reference image available for each class.

**Pretrained architectures:**
- ResNet-50 pretrained on Places365 for scene recognition.
- SigLIP 2 Base (ViT-B/16) pretrained on WebLI.

**Method:** Frozen pretrained feature extraction, reference-only augmentation, weighted prototype refinement, one-shot cosine similarity, per-image score standardization, and fixed equal-weight model fusion.

**One-shot restriction:** Exactly one labeled reference image is used per class, giving 16 labeled reference images in total. All augmented views are derived from these same images. No additional labeled images are introduced.

**Reference augmentation:** Four deterministic views are used: original image, horizontal flip, mild center crop, and mild Cutout. The predetermined prototype weights are 0.70, 0.10, 0.10, and 0.10, respectively.

**Evaluation:** Validation and test images are processed without augmentation. The original-view ensemble serves as the Experiment 2-style baseline, while the refined-prototype ensemble is the primary Experiment 5 method.

**Model fusion:** ResNet-50 and SigLIP 2 contribute equally to the final standardized similarity scores. No model parameters or fusion weights are optimized using validation or test labels.

**Baseline:** Experiment 2 achieved 84.10% validation accuracy and 85.25% test accuracy using the same two pretrained architectures without reference augmentation.

**Implementation:** Both the models remain frozen. Extracted features and results are cached and saved for reproducibility.

### 5.1 Model Initialization and Reference-Only Augmentation

Initializing ResNet-50 using Places365 pretrained weights and SigLIP 2 Base using WebLI pretrained weights.

Removing the classification layer from ResNet-50 and use the visual encoder output from SigLIP 2 to obtain image embeddings.

Keeping both pretrained models frozen and using evaluation mode throughout the experiment.

Generating four views from each of the 16 labeled reference images:

1. Original image — weight 0.70.
2. Horizontal flip — weight 0.10.
3. Mild center crop — weight 0.10.
4. Mild Cutout — weight 0.10.

Applying Cutout by masking a small rectangular region covering approximately 8% of the image area. Using a fixed random seed to ensure reproducibility.

In [15]:
# ============================================================
# EXPERIMENT 5 — INITIALIZATION AND REFERENCE AUGMENTATION
# ResNet-50 Places365 + SigLIP 2 Base
# ============================================================

import sys
import subprocess
import gc
import json
import time
import random
import hashlib
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from PIL import Image
from torchvision import transforms
from torchvision.transforms import functional as TF
from IPython.display import display
from torch.utils.data import Dataset, DataLoader

try:
    import open_clip
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "-q", "open_clip_torch"
    ])
    import open_clip

# ------------------------------------------------------------
# 1. Experiment configuration.
# ------------------------------------------------------------

E5_START = time.time()

E5_DIR = ROOT / "experiment_5"
E5_DIR.mkdir(parents=True, exist_ok=True)

E5_CACHE_DIR = E5_DIR / "feature_cache"
E5_CACHE_DIR.mkdir(parents=True, exist_ok=True)

E5_SEED = 42
E5_BATCH_SIZE = 16

random.seed(E5_SEED)
np.random.seed(E5_SEED)
torch.manual_seed(E5_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(E5_SEED)

E5_DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

E5_RESNET_ID = "resnet50_places365"
E5_SIGLIP_ID = "ViT-B-16-SigLIP2"
E5_SIGLIP_WEIGHTS = "webli"

E5_VIEW_NAMES = [
    "original",
    "horizontal_flip",
    "mild_center_crop",
    "mild_cutout"
]

E5_VIEW_WEIGHTS = torch.tensor(
    [0.70, 0.10, 0.10, 0.10],
    dtype=torch.float32
)

assert len(E5_VIEW_NAMES) == 4
assert abs(
    E5_VIEW_WEIGHTS.sum().item() - 1.0
) < 1e-6

# ------------------------------------------------------------
# 2. Validating common setup.
# ------------------------------------------------------------

assert len(CLASS_NAMES) == 16
assert len(ONE_SHOT_SAMPLES) == 16
assert len(VALIDATION_SAMPLES) == 2384
assert len(TEST_SAMPLES) == 400

def e5_validate_samples(samples, name):
    for path, label in samples:
        assert Path(path).is_file(), (
            f"{name}: missing image {path}"
        )
        assert 0 <= int(label) < 16

e5_validate_samples(
    ONE_SHOT_SAMPLES, "Reference"
)
e5_validate_samples(
    VALIDATION_SAMPLES, "Validation"
)
e5_validate_samples(
    TEST_SAMPLES, "Test"
)

e5_reference_labels_check = sorted(
    int(label) for _, label in ONE_SHOT_SAMPLES
)

assert e5_reference_labels_check == list(range(16))

def e5_path_set(samples):
    return {
        str(Path(path).resolve())
        for path, _ in samples
    }

assert e5_path_set(ONE_SHOT_SAMPLES).isdisjoint(
    e5_path_set(VALIDATION_SAMPLES)
)

assert e5_path_set(ONE_SHOT_SAMPLES).isdisjoint(
    e5_path_set(TEST_SAMPLES)
)

assert e5_path_set(VALIDATION_SAMPLES).isdisjoint(
    e5_path_set(TEST_SAMPLES)
)

# ------------------------------------------------------------
# 3. Downloading ResNet-50 Places365 checkpoint.
# ------------------------------------------------------------

E5_RESNET_URL = (
    "http://places2.csail.mit.edu/"
    "models_places365/resnet50_places365.pth.tar"
)

E5_RESNET_FILE = (
    E5_DIR / "resnet50_places365.pth.tar"
)

# Reuse a local checkpoint if it already exists.
# This is optional and does not depend on E4 variables.

e5_existing_checkpoints = [
    E5_RESNET_FILE,
    ROOT / "experiment_4" /
    "resnet50_places365.pth.tar",
    ROOT / "experiment_2" /
    "resnet50_places365.pth.tar"
]

e5_checkpoint_path = None

for candidate in e5_existing_checkpoints:
    if (
        candidate.exists()
        and candidate.stat().st_size > 10_000_000
    ):
        e5_checkpoint_path = candidate
        break

if e5_checkpoint_path is None:

    print("Downloading ResNet-50 Places365...")

    temporary_path = E5_RESNET_FILE.with_suffix(
        ".partial"
    )

    temporary_path.unlink(missing_ok=True)

    try:
        urllib.request.urlretrieve(
            E5_RESNET_URL,
            str(temporary_path)
        )

        if temporary_path.stat().st_size < 10_000_000:
            raise RuntimeError(
                "Places365 checkpoint is unexpectedly small."
            )

        temporary_path.replace(E5_RESNET_FILE)

    except Exception:
        temporary_path.unlink(missing_ok=True)
        raise

    e5_checkpoint_path = E5_RESNET_FILE

print("Places365 checkpoint:", e5_checkpoint_path)

# ------------------------------------------------------------
# 4. Initializing ResNet-50 Places365.
# ------------------------------------------------------------

e5_resnet = models.resnet50(
    weights=None,
    num_classes=365
)

e5_checkpoint = torch.load(
    e5_checkpoint_path,
    map_location="cpu",
    weights_only=False
)

e5_state = e5_checkpoint.get(
    "state_dict",
    e5_checkpoint
)

e5_state = {
    key.removeprefix("module."): value
    for key, value in e5_state.items()
}

e5_resnet.load_state_dict(
    e5_state,
    strict=True
)

e5_resnet.fc = nn.Identity()
e5_resnet.eval()
e5_resnet.requires_grad_(False)

del e5_checkpoint, e5_state

print("ResNet-50 Places365 initialized.")

# ------------------------------------------------------------
# 5. Initializing SigLIP 2 Base.
# ------------------------------------------------------------

e5_siglip, _, e5_siglip_transform = (
    open_clip.create_model_and_transforms(
        E5_SIGLIP_ID,
        pretrained=E5_SIGLIP_WEIGHTS,
        device="cpu"
    )
)

e5_siglip.eval()
e5_siglip.requires_grad_(False)

print("SigLIP 2 Base initialized.")

# ------------------------------------------------------------
# 6. Model preprocessing.
# ------------------------------------------------------------

E5_IMAGENET_MEAN = [0.485, 0.456, 0.406]
E5_IMAGENET_STD = [0.229, 0.224, 0.225]

# Match the original ResNet-50 evaluation preprocessing
# used in the previous experiment implementation.

e5_resnet_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        E5_IMAGENET_MEAN,
        E5_IMAGENET_STD
    )
])

# SigLIP 2 uses its own pretrained preprocessing,
# provided by OpenCLIP.

# ------------------------------------------------------------
# 7. Deterministic reference-only augmentations.
# ------------------------------------------------------------

def e5_create_reference_views(image, sample_index):
    image = image.convert("RGB")

    width, height = image.size

    # View 1: Original.
    original = image.copy()

    # View 2: Horizontal flip.
    horizontal_flip = TF.hflip(image)

    # View 3: Mild center crop (90%).
    crop_width = max(
        1, round(width * 0.90)
    )

    crop_height = max(
        1, round(height * 0.90)
    )

    crop_left = (width - crop_width) // 2
    crop_top = (height - crop_height) // 2

    mild_crop = TF.crop(
        image,
        crop_top,
        crop_left,
        crop_height,
        crop_width
    )

    # View 4: Mild Cutout.
    # Mask approximately 8% of the image area.
    # Use an image-specific deterministic seed.

    cutout = image.copy()

    mask_fraction = 0.08
    mask_side_fraction = mask_fraction ** 0.5

    mask_width = max(
        1, round(width * mask_side_fraction)
    )

    mask_height = max(
        1, round(height * mask_side_fraction)
    )

    rng = random.Random(
        E5_SEED * 100000 + sample_index
    )

    mask_left = rng.randint(
        0,
        max(0, width - mask_width)
    )

    mask_top = rng.randint(
        0,
        max(0, height - mask_height)
    )

    # Use ImageNet mean-color filling rather than
    # an artificial black square.
    fill_color = (
        124, 116, 104
    )

    cutout.paste(
        Image.new(
            "RGB",
            (mask_width, mask_height),
            fill_color
        ),
        (
            mask_left,
            mask_top
        )
    )

    return [
        original,
        horizontal_flip,
        mild_crop,
        cutout
    ]


# ------------------------------------------------------------
# 8. Dataset for reference views.
# ------------------------------------------------------------

class E5ReferenceDataset(Dataset):

    def __init__(self, samples, transform):
        self.samples = list(samples)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]

        with Image.open(path) as source:
            image = source.convert("RGB")

        views = e5_create_reference_views(
            image,
            sample_index=index
        )

        tensors = torch.stack([
            self.transform(view)
            for view in views
        ], dim=0)

        return (
            tensors,
            int(label)
        )


# ------------------------------------------------------------
# 9. Dataset for original-view evaluation.
# ------------------------------------------------------------

class E5EvaluationDataset(Dataset):

    def __init__(self, samples, transform):
        self.samples = list(samples)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]

        with Image.open(path) as source:
            image = source.convert("RGB")

        return (
            self.transform(image),
            int(label)
        )


# ------------------------------------------------------------
# 10. Feature-cache identity.
# ------------------------------------------------------------

e5_split_info = [
    sorted(
        [
            (
                str(Path(path).resolve()),
                int(label)
            )
            for path, label in samples
        ]
    )
    for samples in [
        ONE_SHOT_SAMPLES,
        VALIDATION_SAMPLES,
        TEST_SAMPLES
    ]
]

E5_CACHE_ID = hashlib.sha256(
    json.dumps({
        "splits": e5_split_info,
        "resnet": E5_RESNET_ID,
        "siglip": E5_SIGLIP_ID,
        "siglip_weights": E5_SIGLIP_WEIGHTS,
        "views": E5_VIEW_NAMES,
        "weights": E5_VIEW_WEIGHTS.tolist(),
        "cutout_seed": E5_SEED,
        "cutout_fraction": 0.08,
        "crop_fraction": 0.90,
        "version": "E5_reference_only_v1"
    }, sort_keys=True).encode()
).hexdigest()[:16]

# ------------------------------------------------------------
# 11. Verifying frozen parameters.
# ------------------------------------------------------------

assert not any(
    parameter.requires_grad
    for parameter in e5_resnet.parameters()
)

assert not any(
    parameter.requires_grad
    for parameter in e5_siglip.parameters()
)

print("\n" + "=" * 65)
print("EXPERIMENT 5 — INITIALIZATION COMPLETE")
print("=" * 65)

print("Architecture: ResNet-50 + SigLIP 2")
print("Reference images:", len(ONE_SHOT_SAMPLES))
print("Validation images:", len(VALIDATION_SAMPLES))
print("Test images:", len(TEST_SAMPLES))
print("Reference views:", E5_VIEW_NAMES)
print("Prototype weights:", E5_VIEW_WEIGHTS.tolist())
print("Fusion: 50% ResNet + 50% SigLIP")
print("Fine-tuning: None")
print("Device:", E5_DEVICE)
print("Cache ID:", E5_CACHE_ID)
print("Output directory:", E5_DIR)

Places365 checkpoint: /content/assignment3/experiment_4/resnet50_places365.pth.tar
ResNet-50 Places365 initialized.
SigLIP 2 Base initialized.

EXPERIMENT 5 — INITIALIZATION COMPLETE
Architecture: ResNet-50 + SigLIP 2
Reference images: 16
Validation images: 2384
Test images: 400
Reference views: ['original', 'horizontal_flip', 'mild_center_crop', 'mild_cutout']
Prototype weights: [0.699999988079071, 0.10000000149011612, 0.10000000149011612, 0.10000000149011612]
Fusion: 50% ResNet + 50% SigLIP
Fine-tuning: None
Device: cuda
Cache ID: 1eb3ccf2f1cacabd
Output directory: /content/assignment3/experiment_5


### 5.2 Reference Prototype Refinement and Validation Evaluation

Extracting embeddings for the original and three augmented views of each of the 16 labeled reference images using both frozen pretrained models.

Normalizing the embeddings individually and construct refined class prototypes using fixed weights of 0.70 for the original image and 0.10 each for the horizontal flip, mild center crop, and Cutout views. Normalizing the resulting prototypes again.

Extracting original-view embeddings for the validation images without applying augmentation.

Calculating cosine similarity between each validation embedding and the 16 class prototypes.

Standardizing similarity scores independently for each model and image, then combining ResNet-50 and SigLIP 2 using fixed equal weights.

Evaluating both the original-reference baseline and refined-prototype approach to measure the effect of reference-only augmentation. Reporting validation accuracies for diagnostic analysis without using them to optimize model parameters or augmentation settings.

In [16]:
# ============================================================
# EXPERIMENT 5 — REFERENCE PROTOTYPES AND VALIDATION
# ============================================================

def e5_forward(model, model_name, images):
    if model_name == "siglip":
        return model.encode_image(images)
    return model(images)


def e5_extract_embeddings(
    model,
    model_name,
    samples,
    transform,
    reference_views=False
):
    if reference_views:
        dataset = E5ReferenceDataset(
            samples,
            transform
        )
    else:
        dataset = E5EvaluationDataset(
            samples,
            transform
        )

    loader = DataLoader(
        dataset,
        batch_size=E5_BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=(
            E5_DEVICE.type == "cuda"
        )
    )

    all_features = []
    all_labels = []

    model = model.to(E5_DEVICE).eval()

    try:
        with torch.inference_mode():

            for batch_index, (images, labels) in enumerate(
                loader
            ):

                if reference_views:
                    # Shape: [batch, 4, 3, H, W]
                    batch_size_actual, num_views = (
                        images.shape[:2]
                    )

                    assert num_views == 4

                    view_features = []

                    for view_index in range(num_views):

                        view_images = images[
                            :, view_index
                        ].to(
                            E5_DEVICE,
                            non_blocking=True
                        )

                        if E5_DEVICE.type == "cuda":
                            with torch.autocast(
                                device_type="cuda",
                                dtype=torch.float16
                            ):
                                embeddings = e5_forward(
                                    model,
                                    model_name,
                                    view_images
                                )
                        else:
                            embeddings = e5_forward(
                                model,
                                model_name,
                                view_images
                            )

                        embeddings = F.normalize(
                            embeddings.float(),
                            p=2,
                            dim=-1
                        )

                        view_features.append(
                            embeddings.cpu()
                        )

                    features = torch.stack(
                        view_features,
                        dim=1
                    )

                else:
                    images = images.to(
                        E5_DEVICE,
                        non_blocking=True
                    )

                    if E5_DEVICE.type == "cuda":
                        with torch.autocast(
                            device_type="cuda",
                            dtype=torch.float16
                        ):
                            embeddings = e5_forward(
                                model,
                                model_name,
                                images
                            )
                    else:
                        embeddings = e5_forward(
                            model,
                            model_name,
                            images
                        )

                    features = F.normalize(
                        embeddings.float(),
                        p=2,
                        dim=-1
                    ).cpu()

                all_features.append(features)
                all_labels.append(labels.cpu())

                if (batch_index + 1) % 25 == 0:
                    processed = min(
                        (batch_index + 1)
                        * E5_BATCH_SIZE,
                        len(dataset)
                    )

                    print(
                        f"{model_name}: "
                        f"{processed}/{len(dataset)}"
                    )

    finally:
        model.cpu()
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    features = torch.cat(
        all_features,
        dim=0
    )

    labels = torch.cat(
        all_labels,
        dim=0
    )

    assert features.shape[0] == len(samples)
    assert torch.isfinite(features).all()

    return features, labels


# ------------------------------------------------------------
# 1. Feature caching.
# ------------------------------------------------------------

def e5_get_cached_embeddings(
    model,
    model_name,
    split_name,
    samples,
    transform,
    reference_views=False
):
    cache_file = (
        E5_CACHE_DIR
        / f"{model_name}_{split_name}_{E5_CACHE_ID}.pt"
    )

    expected_labels = torch.tensor(
        [int(label) for _, label in samples],
        dtype=torch.long
    )

    expected_ndim = (
        3 if reference_views else 2
    )

    if cache_file.exists():
        try:
            cached = torch.load(
                cache_file,
                map_location="cpu",
                weights_only=True
            )

            features = cached["features"]
            labels = cached["labels"]

            valid = (
                features.ndim == expected_ndim
                and features.shape[0] == len(samples)
                and torch.equal(
                    labels,
                    expected_labels
                )
                and torch.isfinite(features).all()
            )

            if reference_views:
                valid = (
                    valid
                    and features.shape[1] == 4
                )

            if valid:
                print(
                    "Loaded cached embeddings:",
                    model_name,
                    split_name
                )

                return features, labels

        except Exception as error:
            print(
                "Cache unreadable; recomputing:",
                str(error)
            )

    print(
        "\nExtracting embeddings:",
        model_name,
        split_name
    )

    features, labels = e5_extract_embeddings(
        model=model,
        model_name=model_name,
        samples=samples,
        transform=transform,
        reference_views=reference_views
    )

    assert torch.equal(
        labels,
        expected_labels
    )

    temporary_file = cache_file.with_suffix(
        ".partial"
    )

    torch.save(
        {
            "features": features,
            "labels": labels
        },
        temporary_file
    )

    temporary_file.replace(cache_file)

    print("Saved cache:", cache_file.name)

    return features, labels


# ------------------------------------------------------------
# 2. Building original and refined class prototypes.
# ------------------------------------------------------------

def e5_construct_prototypes(
    reference_features,
    reference_labels
):
    assert reference_features.ndim == 3
    assert reference_features.shape[1] == 4

    order = torch.argsort(
        reference_labels
    )

    ordered_labels = reference_labels[order]

    assert torch.equal(
        ordered_labels,
        torch.arange(16)
    )

    features = F.normalize(
        reference_features[order].float(),
        p=2,
        dim=-1
    )

    # Original-view prototypes:
    # Equivalent to using only the reference image.
    original_prototypes = features[:, 0, :]

    # Refined prototypes:
    # 70% original + 10% each augmented view.
    weights = E5_VIEW_WEIGHTS.view(
        1, 4, 1
    )

    refined_prototypes = (
        features * weights
    ).sum(dim=1)

    refined_prototypes = F.normalize(
        refined_prototypes,
        p=2,
        dim=-1
    )

    return (
        original_prototypes,
        refined_prototypes
    )


# ------------------------------------------------------------
# 3. Similarity and fusion.
# ------------------------------------------------------------

def e5_cosine_scores(
    evaluation_features,
    prototypes
):
    evaluation_features = F.normalize(
        evaluation_features.float(),
        p=2,
        dim=-1
    )

    prototypes = F.normalize(
        prototypes.float(),
        p=2,
        dim=-1
    )

    return (
        evaluation_features
        @ prototypes.T
    )


def e5_standardize_scores(scores):
    mean = scores.mean(
        dim=1,
        keepdim=True
    )

    std = scores.std(
        dim=1,
        keepdim=True,
        unbiased=False
    ).clamp_min(1e-6)

    return (scores - mean) / std


def e5_fuse_scores(
    resnet_scores,
    siglip_scores
):
    return (
        0.5 * e5_standardize_scores(
            resnet_scores
        )
        +
        0.5 * e5_standardize_scores(
            siglip_scores
        )
    )


def e5_accuracy(scores, labels):
    predictions = scores.argmax(
        dim=1
    )

    return (
        (predictions == labels)
        .float()
        .mean()
        .item() * 100
    )


# ------------------------------------------------------------
# 4. Model configuration.
# ------------------------------------------------------------

E5_MODELS = {
    "resnet": {
        "model": e5_resnet,
        "transform": e5_resnet_transform
    },
    "siglip": {
        "model": e5_siglip,
        "transform": e5_siglip_transform
    }
}

E5_METHODS = [
    "original",
    "refined"
]

e5_prototypes = {}
e5_validation_features = {}
e5_validation_scores = {
    "original": {},
    "refined": {}
}

e5_validation_accuracies = {
    "original": {},
    "refined": {}
}

e5_validation_labels = torch.tensor(
    [
        int(label)
        for _, label in VALIDATION_SAMPLES
    ],
    dtype=torch.long
)


# ------------------------------------------------------------
# 5. Extracting reference and validation embeddings.
# ------------------------------------------------------------

for model_name, config in E5_MODELS.items():

    print("\n" + "=" * 60)
    print("MODEL:", model_name.upper())
    print("=" * 60)

    reference_features, reference_labels = (
        e5_get_cached_embeddings(
            model=config["model"],
            model_name=model_name,
            split_name="reference_views",
            samples=ONE_SHOT_SAMPLES,
            transform=config["transform"],
            reference_views=True
        )
    )

    original_proto, refined_proto = (
        e5_construct_prototypes(
            reference_features,
            reference_labels
        )
    )

    e5_prototypes[
        (model_name, "original")
    ] = original_proto

    e5_prototypes[
        (model_name, "refined")
    ] = refined_proto

    validation_features, validation_labels = (
        e5_get_cached_embeddings(
            model=config["model"],
            model_name=model_name,
            split_name="validation_original",
            samples=VALIDATION_SAMPLES,
            transform=config["transform"],
            reference_views=False
        )
    )

    assert torch.equal(
        validation_labels,
        e5_validation_labels
    )

    e5_validation_features[
        model_name
    ] = validation_features

    for method in E5_METHODS:

        scores = e5_cosine_scores(
            validation_features,
            e5_prototypes[
                (model_name, method)
            ]
        )

        e5_validation_scores[
            method
        ][model_name] = scores

        e5_validation_accuracies[
            method
        ][model_name] = e5_accuracy(
            scores,
            e5_validation_labels
        )


# ------------------------------------------------------------
# 6. Fixed equal-weight fusion.
# ------------------------------------------------------------

e5_fused_validation_scores = {}
e5_fused_validation_accuracies = {}

for method in E5_METHODS:

    fused_scores = e5_fuse_scores(
        e5_validation_scores[
            method
        ]["resnet"],
        e5_validation_scores[
            method
        ]["siglip"]
    )

    e5_fused_validation_scores[
        method
    ] = fused_scores

    e5_fused_validation_accuracies[
        method
    ] = e5_accuracy(
        fused_scores,
        e5_validation_labels
    )


# ------------------------------------------------------------
# 7. Displaying validation diagnostics.
# ------------------------------------------------------------

e5_validation_rows = []

for model_name in ["resnet", "siglip"]:

    original_accuracy = (
        e5_validation_accuracies[
            "original"
        ][model_name]
    )

    refined_accuracy = (
        e5_validation_accuracies[
            "refined"
        ][model_name]
    )

    e5_validation_rows.append({
        "Model": model_name.upper(),
        "Original Validation (%)": round(
            original_accuracy, 2
        ),
        "Refined Validation (%)": round(
            refined_accuracy, 2
        ),
        "Change (pp)": round(
            refined_accuracy - original_accuracy,
            2
        )
    })

e5_original_val = (
    e5_fused_validation_accuracies["original"]
)

e5_refined_val = (
    e5_fused_validation_accuracies["refined"]
)

e5_validation_rows.append({
    "Model": "TWO-MODEL ENSEMBLE",
    "Original Validation (%)": round(
        e5_original_val, 2
    ),
    "Refined Validation (%)": round(
        e5_refined_val, 2
    ),
    "Change (pp)": round(
        e5_refined_val - e5_original_val,
        2
    )
})

e5_validation_df = pd.DataFrame(
    e5_validation_rows
)

e5_validation_df.to_csv(
    E5_DIR / "experiment_5_validation_diagnostics.csv",
    index=False
)

print("\n" + "=" * 65)
print("EXPERIMENT 5 — VALIDATION RESULTS")
print("=" * 65)

display(e5_validation_df)

E5_VAL_ACCURACY = e5_refined_val

print(
    f"\nPrimary Refined-Prototype "
    f"Validation Accuracy: "
    f"{E5_VAL_ACCURACY:.2f}%"
)

print(
    "No augmentation weights or model parameters "
    "were optimized using validation labels."
)

print("\nReady for Experiment 5 Code Cell 3.")


MODEL: RESNET

Extracting embeddings: resnet reference_views
Saved cache: resnet_reference_views_1eb3ccf2f1cacabd.pt

Extracting embeddings: resnet validation_original
resnet: 400/2384
resnet: 800/2384
resnet: 1200/2384
resnet: 1600/2384
resnet: 2000/2384
Saved cache: resnet_validation_original_1eb3ccf2f1cacabd.pt

MODEL: SIGLIP

Extracting embeddings: siglip reference_views
Saved cache: siglip_reference_views_1eb3ccf2f1cacabd.pt

Extracting embeddings: siglip validation_original
siglip: 400/2384
siglip: 800/2384
siglip: 1200/2384
siglip: 1600/2384
siglip: 2000/2384
Saved cache: siglip_validation_original_1eb3ccf2f1cacabd.pt

EXPERIMENT 5 — VALIDATION RESULTS


,Model,Original Validation (%),Refined Validation (%),Change (pp)
0,RESNET,81.67,82.05,0.38
1,SIGLIP,78.61,78.57,-0.04
2,TWO-MODEL ENSEMBLE,84.10,84.19,0.08



Primary Refined-Prototype Validation Accuracy: 84.19%
No augmentation weights or model parameters were optimized using validation labels.

Ready for Experiment 5 Code Cell 3.


### 5.3 Test Evaluation and Final Results

Evaluating the frozen ResNet-50 Places365 and SigLIP 2 models on the separate 400-image test dataset using original-view preprocessing.

Calculating cosine similarity between each test embedding and the original or refined class prototypes.

Standardizing class similarity scores independently for each model and image, then combining both models using fixed equal weights.

Reporting individual model test accuracies and comparing the original-reference ensemble with the refined-prototype ensemble.

The primary Experiment 5 result is the refined-prototype ensemble test accuracy. The original-reference ensemble serves as an ablation baseline.

Saving the validation and test diagnostics, experiment results, predictions, prototype embeddings, and configuration for reproducibility.

Comparing Experiment 5 with Experiment 2 to determine whether reference-only augmentation improves one-shot classification without fine-tuning or introducing additional labeled training images.

In [17]:
# ============================================================
# EXPERIMENT 5 — TEST EVALUATION AND FINAL RESULTS
# ResNet-50 Places365 + SigLIP 2
# ============================================================

# ------------------------------------------------------------
# 1. Extracting original-view test embeddings.
# ------------------------------------------------------------

e5_test_features = {}

e5_test_labels = torch.tensor(
    [
        int(label)
        for _, label in TEST_SAMPLES
    ],
    dtype=torch.long
)

for model_name, config in E5_MODELS.items():

    print("\n" + "=" * 60)
    print("TEST MODEL:", model_name.upper())
    print("=" * 60)

    features, labels = e5_get_cached_embeddings(
        model=config["model"],
        model_name=model_name,
        split_name="test_original",
        samples=TEST_SAMPLES,
        transform=config["transform"],
        reference_views=False
    )

    assert torch.equal(
        labels,
        e5_test_labels
    )

    e5_test_features[
        model_name
    ] = features


# ------------------------------------------------------------
# 2. Computing individual model test scores.
# ------------------------------------------------------------

e5_test_scores = {
    "original": {},
    "refined": {}
}

e5_test_accuracies = {
    "original": {},
    "refined": {}
}

for method in E5_METHODS:

    for model_name in E5_MODELS:

        scores = e5_cosine_scores(
            e5_test_features[model_name],
            e5_prototypes[
                (model_name, method)
            ]
        )

        e5_test_scores[
            method
        ][model_name] = scores

        e5_test_accuracies[
            method
        ][model_name] = e5_accuracy(
            scores,
            e5_test_labels
        )


# ------------------------------------------------------------
# 3. Fixed 50/50 ensemble fusion.
# ------------------------------------------------------------

e5_fused_test_scores = {}
e5_fused_test_accuracies = {}
e5_test_predictions = {}

for method in E5_METHODS:

    fused_scores = e5_fuse_scores(
        e5_test_scores[
            method
        ]["resnet"],
        e5_test_scores[
            method
        ]["siglip"]
    )

    e5_fused_test_scores[
        method
    ] = fused_scores

    e5_test_predictions[
        method
    ] = fused_scores.argmax(
        dim=1
    )

    e5_fused_test_accuracies[
        method
    ] = e5_accuracy(
        fused_scores,
        e5_test_labels
    )


# ------------------------------------------------------------
# 4. Calculating final experiment accuracies.
# ------------------------------------------------------------

E5_ORIGINAL_VAL_ACCURACY = (
    e5_fused_validation_accuracies["original"]
)

E5_ORIGINAL_TEST_ACCURACY = (
    e5_fused_test_accuracies["original"]
)

E5_VAL_ACCURACY = (
    e5_fused_validation_accuracies["refined"]
)

E5_TEST_ACCURACY = (
    e5_fused_test_accuracies["refined"]
)

E5_RUNTIME = time.time() - E5_START

E5_CORRECT = int(
    (
        e5_test_predictions["refined"]
        == e5_test_labels
    ).sum().item()
)

# ------------------------------------------------------------
# 5. Building complete diagnostic table.
# ------------------------------------------------------------

e5_diagnostic_rows = []

for model_name in ["resnet", "siglip"]:

    for method in E5_METHODS:

        e5_diagnostic_rows.append({
            "Experiment": "E5",
            "Model": model_name.upper(),
            "Prototype": method,
            "Validation Accuracy (%)": round(
                e5_validation_accuracies[
                    method
                ][model_name],
                2
            ),
            "Test Accuracy (%)": round(
                e5_test_accuracies[
                    method
                ][model_name],
                2
            )
        })

for method in E5_METHODS:

    e5_diagnostic_rows.append({
        "Experiment": "E5",
        "Model": "TWO-MODEL ENSEMBLE",
        "Prototype": method,
        "Validation Accuracy (%)": round(
            e5_fused_validation_accuracies[
                method
            ],
            2
        ),
        "Test Accuracy (%)": round(
            e5_fused_test_accuracies[
                method
            ],
            2
        )
    })

e5_diagnostics_df = pd.DataFrame(
    e5_diagnostic_rows
)

e5_diagnostics_df.to_csv(
    E5_DIR / "experiment_5_diagnostics.csv",
    index=False
)

print("\nMODEL AND ENSEMBLE DIAGNOSTICS")
display(e5_diagnostics_df)


# ------------------------------------------------------------
# 6. Measuring the effect of prototype refinement.
# ------------------------------------------------------------

e5_effect_df = pd.DataFrame([
    {
        "Evaluation": "Validation",
        "Original (%)": round(
            E5_ORIGINAL_VAL_ACCURACY,
            2
        ),
        "Refined (%)": round(
            E5_VAL_ACCURACY,
            2
        ),
        "Change (pp)": round(
            E5_VAL_ACCURACY
            - E5_ORIGINAL_VAL_ACCURACY,
            2
        )
    },
    {
        "Evaluation": "Test",
        "Original (%)": round(
            E5_ORIGINAL_TEST_ACCURACY,
            2
        ),
        "Refined (%)": round(
            E5_TEST_ACCURACY,
            2
        ),
        "Change (pp)": round(
            E5_TEST_ACCURACY
            - E5_ORIGINAL_TEST_ACCURACY,
            2
        )
    }
])

e5_effect_df.to_csv(
    E5_DIR / "experiment_5_refinement_effect.csv",
    index=False
)

print("\nPROTOTYPE REFINEMENT EFFECT")
display(e5_effect_df)


# ------------------------------------------------------------
# 7. Preparing primary experiment result.
# ------------------------------------------------------------

E5_ARCHITECTURE = (
    "ResNet-50 (Places365) + "
    "SigLIP 2 Base (ViT-B/16, WebLI)"
)

E5_METHOD = (
    "One-shot cosine similarity + "
    "reference-only prototype refinement "
    "(original, flip, mild crop, Cutout) + "
    "fixed 50/50 fusion"
)

e5_results_df = pd.DataFrame([{
    "Experiment": "E5",
    "Architecture": E5_ARCHITECTURE,
    "Method": E5_METHOD,
    "Validation Accuracy (%)": round(
        E5_VAL_ACCURACY, 2
    ),
    "Test Accuracy (%)": round(
        E5_TEST_ACCURACY, 2
    ),
    "Runtime (seconds)": round(
        E5_RUNTIME, 2
    )
}])

e5_results_df.to_csv(
    E5_DIR / "experiment_5_results.csv",
    index=False
)


# ------------------------------------------------------------
# 8. Saving class prototypes and predictions.
# ------------------------------------------------------------

torch.save(
    {
        "prototypes": e5_prototypes,
        "test_labels": e5_test_labels,
        "original_predictions": (
            e5_test_predictions["original"]
        ),
        "refined_predictions": (
            e5_test_predictions["refined"]
        ),
        "original_scores": (
            e5_fused_test_scores["original"]
        ),
        "refined_scores": (
            e5_fused_test_scores["refined"]
        ),
        "class_names": list(CLASS_NAMES),
        "view_names": E5_VIEW_NAMES,
        "view_weights": E5_VIEW_WEIGHTS,
        "cache_identity": E5_CACHE_ID
    },
    E5_DIR / "experiment_5_predictions.pt"
)


# ------------------------------------------------------------
# 9. Saving per-image test predictions.
# ------------------------------------------------------------

e5_prediction_rows = []

original_predictions = (
    e5_test_predictions["original"].tolist()
)

refined_predictions = (
    e5_test_predictions["refined"].tolist()
)

for index, (path, label) in enumerate(
    TEST_SAMPLES
):
    original_pred = original_predictions[index]
    refined_pred = refined_predictions[index]

    e5_prediction_rows.append({
        "Image": str(path),
        "Ground Truth": CLASS_NAMES[int(label)],
        "Original Prediction": (
            CLASS_NAMES[original_pred]
        ),
        "Refined Prediction": (
            CLASS_NAMES[refined_pred]
        ),
        "Original Correct": (
            int(original_pred == int(label))
        ),
        "Refined Correct": (
            int(refined_pred == int(label))
        )
    })

e5_predictions_df = pd.DataFrame(
    e5_prediction_rows
)

e5_predictions_df.to_csv(
    E5_DIR / "experiment_5_test_predictions.csv",
    index=False
)


# ------------------------------------------------------------
# 10. Saving reproducibility configuration.
# ------------------------------------------------------------

e5_config = {
    "experiment": "E5",
    "architecture": E5_ARCHITECTURE,
    "method": E5_METHOD,
    "seed": E5_SEED,
    "reference_images_per_class": 1,
    "total_reference_images": 16,
    "validation_images": len(
        VALIDATION_SAMPLES
    ),
    "test_images": len(
        TEST_SAMPLES
    ),
    "models": {
        "resnet": E5_RESNET_ID,
        "siglip": E5_SIGLIP_ID,
        "siglip_weights": E5_SIGLIP_WEIGHTS
    },
    "frozen_models": True,
    "fine_tuning": False,
    "training_epochs": 0,
    "reference_augmentation": {
        "views": E5_VIEW_NAMES,
        "weights": E5_VIEW_WEIGHTS.tolist(),
        "crop_fraction": 0.90,
        "cutout_area_fraction": 0.08,
        "cutout_seed": E5_SEED,
        "aggregation": (
            "Normalize each embedding, "
            "weighted average, normalize again"
        )
    },
    "evaluation_augmentation": False,
    "similarity": "cosine",
    "score_normalization": "per-image z-score",
    "model_fusion": {
        "resnet": 0.5,
        "siglip": 0.5
    },
    "original_validation_accuracy": (
        E5_ORIGINAL_VAL_ACCURACY
    ),
    "original_test_accuracy": (
        E5_ORIGINAL_TEST_ACCURACY
    ),
    "refined_validation_accuracy": (
        E5_VAL_ACCURACY
    ),
    "refined_test_accuracy": (
        E5_TEST_ACCURACY
    ),
    "runtime_seconds": E5_RUNTIME,
    "cache_identity": E5_CACHE_ID
}

with open(
    E5_DIR / "experiment_5_config.json",
    "w"
) as file:
    json.dump(
        e5_config,
        file,
        indent=2
    )


# ------------------------------------------------------------
# 11. Updating common experiment tracking.
# ------------------------------------------------------------

record_experiment(
    experiment="E5",
    architecture=E5_ARCHITECTURE,
    method=E5_METHOD,
    validation_accuracy=E5_VAL_ACCURACY,
    test_accuracy=E5_TEST_ACCURACY,
    runtime_seconds=E5_RUNTIME,
    observations=(
        "Two frozen pretrained models; "
        "mild reference-only augmentation "
        "with horizontal flip, center crop "
        "and Cutout; fixed prototype weights "
        "70/10/10/10; no fine-tuning; "
        "original-view evaluation; "
        "fixed 50/50 fusion."
    )
)


# ------------------------------------------------------------
# 12. Final comparison against Experiment 2.
# ------------------------------------------------------------

E5_E2_TEST_BASELINE = 85.25

e5_comparison_df = pd.DataFrame([
    {
        "Experiment": "E2",
        "Method": "Original one-shot prototypes",
        "Test Accuracy (%)": 85.25
    },
    {
        "Experiment": "E5",
        "Method": "Refined reference prototypes",
        "Test Accuracy (%)": round(
            E5_TEST_ACCURACY, 2
        )
    }
])

print("\n" + "=" * 65)
print("EXPERIMENT 5 — FINAL RESULTS")
print("=" * 65)

display(e5_results_df)

print(
    f"\nOriginal-Prototype Validation Accuracy: "
    f"{E5_ORIGINAL_VAL_ACCURACY:.2f}%"
)

print(
    f"Original-Prototype Test Accuracy: "
    f"{E5_ORIGINAL_TEST_ACCURACY:.2f}%"
)

print(
    f"\nRefined-Prototype Validation Accuracy: "
    f"{E5_VAL_ACCURACY:.2f}%"
)

print(
    f"Refined-Prototype Test Accuracy: "
    f"{E5_TEST_ACCURACY:.2f}%"
)

print(
    f"Correct Test Predictions: "
    f"{E5_CORRECT}/{len(TEST_SAMPLES)}"
)

print(
    f"\nPrototype Refinement Test Change: "
    f"{E5_TEST_ACCURACY - E5_ORIGINAL_TEST_ACCURACY:+.2f} "
    "percentage points"
)

print(
    f"Difference from Experiment 2: "
    f"{E5_TEST_ACCURACY - E5_E2_TEST_BASELINE:+.2f} "
    "percentage points"
)

print(
    f"Total Runtime: {E5_RUNTIME:.2f} seconds"
)

print("\nCOMPARISON WITH EXPERIMENT 2")
display(e5_comparison_df)

print("\nExperiment 5 completed successfully.")
print("Results saved in:", E5_DIR)


# ------------------------------------------------------------
# 13. Downloading final result files.
# ------------------------------------------------------------

try:
    from google.colab import files

    for filename in [
        "experiment_5_results.csv",
        "experiment_5_diagnostics.csv",
        "experiment_5_refinement_effect.csv",
        "experiment_5_config.json"
    ]:
        files.download(
            str(E5_DIR / filename)
        )

    common_results_file = (
        ROOT / "experiment_results.csv"
    )

    if common_results_file.exists():
        files.download(
            str(common_results_file)
        )

except ImportError:
    print(
        "Not running in Google Colab. "
        "Results remain saved in:",
        E5_DIR
    )


TEST MODEL: RESNET

Extracting embeddings: resnet test_original
resnet: 400/400
Saved cache: resnet_test_original_1eb3ccf2f1cacabd.pt

TEST MODEL: SIGLIP

Extracting embeddings: siglip test_original
siglip: 400/400
Saved cache: siglip_test_original_1eb3ccf2f1cacabd.pt

MODEL AND ENSEMBLE DIAGNOSTICS


,Experiment,Model,Prototype,Validation Accuracy (%),Test Accuracy (%)
0,E5,RESNET,original,81.67,83.50
1,E5,RESNET,refined,82.05,83.75
2,E5,SIGLIP,original,78.61,78.75
3,E5,SIGLIP,refined,78.57,78.50
4,E5,TWO-MODEL ENSEMBLE,original,84.10,85.25
5,E5,TWO-MODEL ENSEMBLE,refined,84.19,85.00



PROTOTYPE REFINEMENT EFFECT


,Evaluation,Original (%),Refined (%),Change (pp)
0,Validation,84.10,84.19,0.08
1,Test,85.25,85.00,-0.25


,Experiment,Architecture,Method,Validation Accuracy (%),Test Accuracy (%),Runtime (seconds),Observations
0,E4,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + five-view center-...,82.2987,82.75,288.21,Three frozen pretrained models; five determini...
1,E5,ResNet-50 (Places365) + SigLIP 2 Base (ViT-B/1...,One-shot cosine similarity + reference-only pr...,84.1862,85.00,86.35,Two frozen pretrained models; mild reference-o...



EXPERIMENT 5 — FINAL RESULTS


... (10 outputs left out)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Final Model Export: Best-Performing Experiment 2

### Objective
Exporting the model artifacts from Experiment 2, which achieved the highest test accuracy of **85.25% (341/400 correct predictions)**, for standalone inference and reproducibility.

### Method
The final model combines frozen ResNet-50 (Places365) and SigLIP 2 Base (WebLI) encoders using one-shot cosine similarity, per-image score standardization, and fixed equal-weight fusion.

### Exported Artifacts
- **Class prototypes:** L2-normalized reference embeddings for all 16 classes from both pretrained models.
- **Model configuration:** Architecture details, preprocessing settings, class names, and fusion weights.
- **Dataset manifest:** Reference, validation, and test split information.

### Reproducibility
The exported artifacts will be used with the original pretrained model checkpoints to implement standalone inference and reproduce the reported test accuracy locally.

In [23]:
# ============================================================
# FINAL EXPORT — EXPERIMENT 2 MODEL ARTIFACTS
# ============================================================

import json
import zipfile
from pathlib import Path

import torch
import torch.nn.functional as F
from google.colab import files

EXPORT_DIR = ROOT / "final_model_export"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Ensuring Experiment 2's reference features exist.
# If they are not in memory, loading them from its saved cache.

def load_e2_reference(name):
    cache_path = (
        E2_DIR / f"{name}_{E2_CACHE_ID}.pt"
    )

    if not cache_path.exists():
        raise FileNotFoundError(
            f"Missing reference cache: {cache_path}. "
            "Run Experiment 2 Code Cell 2 first."
        )

    saved = torch.load(
        cache_path,
        map_location="cpu",
        weights_only=True
    )

    return saved["features"], saved["labels"]


rn_export, rn_labels = load_e2_reference(
    "resnet_reference"
)

sig_export, sig_labels = load_e2_reference(
    "siglip_reference"
)

# Sorting both sets of embeddings into class-name order.
rn_order = torch.argsort(rn_labels)
sig_order = torch.argsort(sig_labels)

assert torch.equal(
    rn_labels[rn_order],
    torch.arange(16)
)

assert torch.equal(
    sig_labels[sig_order],
    torch.arange(16)
)

resnet_prototypes = F.normalize(
    rn_export[rn_order].float(),
    dim=1
)

siglip_prototypes = F.normalize(
    sig_export[sig_order].float(),
    dim=1
)

# Saving the final model's one-shot prototypes.
torch.save(
    {
        "resnet_prototypes": resnet_prototypes,
        "siglip_prototypes": siglip_prototypes,
        "class_names": list(CLASS_NAMES),
        "resnet_weight": 0.5,
        "siglip_weight": 0.5,
        "cache_identity": E2_CACHE_ID
    },
    EXPORT_DIR / "experiment2_prototypes.pt"
)

# Saving inference configuration.
config = {
    "experiment": "E2",
    "class_names": list(CLASS_NAMES),
    "resnet_architecture": "resnet50",
    "resnet_pretrained": "places365",
    "resnet_checkpoint_url": (
        "http://places2.csail.mit.edu/"
        "models_places365/resnet50_places365.pth.tar"
    ),
    "resnet_resize": [256, 256],
    "resnet_center_crop": 224,
    "resnet_mean": [0.485, 0.456, 0.406],
    "resnet_std": [0.229, 0.224, 0.225],
    "siglip_model": "ViT-B-16-SigLIP2",
    "siglip_pretrained": "webli",
    "siglip_preprocessing": (
        "Use OpenCLIP pretrained evaluation transform"
    ),
    "feature_normalization": "L2",
    "similarity": "cosine",
    "score_normalization": (
        "Per-image z-score using PyTorch "
        "default sample standard deviation"
    ),
    "fusion_weights": {
        "resnet": 0.5,
        "siglip": 0.5
    },
    "test_accuracy": 85.25,
    "test_correct": 341,
    "test_total": 400,
    "cache_identity": E2_CACHE_ID
}

with open(
    EXPORT_DIR / "model_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

# Including the exact one-shot dataset split.
manifest_path = (
    ROOT / "dataset_split_manifest.json"
)

assert manifest_path.exists(), (
    "Dataset split manifest is missing. "
    "Run common setup Cell 7 first."
)

# Package all files into one ZIP.
zip_path = ROOT / "experiment2_final_model.zip"

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:

    for filename in [
        "experiment2_prototypes.pt",
        "model_config.json"
    ]:
        archive.write(
            EXPORT_DIR / filename,
            arcname=f"artifacts/{filename}"
        )

    archive.write(
        manifest_path,
        arcname="dataset_split_manifest.json"
    )

print("=" * 60)
print("EXPERIMENT 2 MODEL EXPORT COMPLETED")
print("=" * 60)

print("Classes:", len(CLASS_NAMES))
print("ResNet prototypes:", resnet_prototypes.shape)
print("SigLIP prototypes:", siglip_prototypes.shape)
print("Best test accuracy: 85.25%")
print("Saved ZIP:", zip_path)

files.download(str(zip_path))

EXPERIMENT 2 MODEL EXPORT COMPLETED
Classes: 16
ResNet prototypes: torch.Size([16, 2048])
SigLIP prototypes: torch.Size([16, 768])
Best test accuracy: 85.25%
Saved ZIP: /content/assignment3/experiment2_final_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>